# Блендинг Session6, Random Forest, CatBoost rich и двух XGBoost

Создайте два Kaggle Dataset из комплектов рядом с ноутбуком и подключите их через Add Input. Файлы в датасетах должны быть доступны в распакованном виде. Внутри каждого датасета нужные файлы лежат в одной папке, которая может быть вложенной.

1. **Session6.** Комплект `session6_blend_input.zip` содержит `comparison_meta.json`, `train_solo_track.parquet`, `test_solo_track.parquet` и `validation_predictions.parquet`.
2. **XGBoost rich k8.** Комплект `xgb_rich_k8_blend_input.zip` содержит `manifest.json`, `protocol.json`, `evaluation_complete.json` и 40 parquet-файлов. В них лежат raw-прогнозы на четыре OOF-недели и локальный test для восьми горизонтов.

Ноутбук найдёт оба набора внутри `/kaggle/input`. Если подключено несколько подходящих версий, укажите папки с файлами в `SESSION_DATA_DIR` и `XGB_DATA_DIR` в ячейке настроек.

Выберите GPU и образ Kaggle с рабочими CUDA, RAPIDS/cuML, CuPy, PyTorch, CatBoost, LightGBM, XGBoost и PyArrow. Запустите Save & Run All. RF, базовые CatBoost, XGBoost на очищенной истории и Gating Network используют GPU. LightGBM Session6, Ridge и небольшие метамодели CatBoost/XGBoost работают на CPU. Длительность зависит от GPU и числа маршрутов в кластерах. Прогнозы RF, CatBoost rich и XGBoost на очищенной истории сохраняются после каждого горизонта. Повторный запуск в той же рабочей директории с теми же входами и кодом переиспользует законченные горизонты. Для возобновления в новой сессии перенесите каталог результата в `/kaggle/working` до запуска.

В локальном тесте оцениваются прогнозы из 10:30 на h1–h8 за 18–31 октября 2025. Все настройки ансамбля выбираются на неделях до этого периода. Архивная оценка уже изучалась в проекте. Эксперимент сохраняет локальные прогнозы и таблицы. Новый конкурсный CSV требует отдельного полного переобучения всех выбранных компонентов.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display


HORIZONS = tuple(range(1, 9))
STEP = pd.Timedelta(minutes=30)
OOF_START = pd.Timestamp('2025-09-20')
LOCAL_TEST_START = pd.Timestamp('2025-10-18')
LOCAL_TEST_END = pd.Timestamp('2025-11-01')
OOF_WINDOWS = list(zip(pd.date_range(OOF_START, periods=4, freq='7D'),
                       pd.date_range(OOF_START + pd.Timedelta(days=7), periods=4, freq='7D')))
KEYS = ['cutoff', 'horizon', 'route_id']
GROUP_KEYS = KEYS[:2]
BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
SEED = 42
THREADS = 4


SESSION_FILES = ('comparison_meta.json', 'train_solo_track.parquet', 'test_solo_track.parquet',
                 'validation_predictions.parquet')
XGB_STAGES = ('oof_week_0', 'oof_week_1', 'oof_week_2', 'calibration_select', 'train_val')
XGB_FILES = ('manifest.json', 'protocol.json', 'evaluation_complete.json',
             *(f'{stage}_h{h}_xgb_k8.parquet' for stage in XGB_STAGES for h in HORIZONS))


def find_dataset(label, path, marker, required):
    if path is None:
        root = Path('/kaggle/input')
        candidates = sorted({file.parent for file in root.rglob(marker)})
        complete = [folder for folder in candidates if all((folder / name).is_file() for name in required)]
        if len(complete) == 1:
            return complete[0]
        if not complete and len(candidates) == 1:
            path = candidates[0]
        else:
            raise FileNotFoundError(f'{label}: нужен один полный набор. Укажите путь к папке в ячейке настроек. '
                                    f'Найденные полные наборы: {complete}; папки с {marker}: {candidates}')
    path = Path(path)
    missing = [name for name in required if not (path / name).is_file()]
    if missing:
        raise FileNotFoundError(f'{label}: в {path} отсутствуют файлы {missing}')
    return path


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, data):
    temp = path.with_suffix('.partial')
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


def save_parquet(path, data):
    temp = path.with_suffix('.partial')
    data.to_parquet(temp, index=False)
    temp.replace(path)


def initialize(session_path, xgb_path):
    global ARTIFACTS, TIMES, ROUTES, VALUES, SESSION_META, SESSION_DATA, XGB_DATA
    SESSION_DATA = find_dataset('Session6', session_path, 'comparison_meta.json', SESSION_FILES)
    XGB_DATA = find_dataset('XGBoost rich k8', xgb_path, 'oof_week_0_h1_xgb_k8.parquet', XGB_FILES)
    train_path = SESSION_DATA / 'train_solo_track.parquet'
    SESSION_META = json.loads((SESSION_DATA / 'comparison_meta.json').read_text())
    xgb_manifest = json.loads((XGB_DATA / 'manifest.json').read_text())
    XGB_PROTOCOL = json.loads((XGB_DATA / 'protocol.json').read_text())
    xgb_evaluation = json.loads((XGB_DATA / 'evaluation_complete.json').read_text())
    contract = dict(code=NOTEBOOK_HASH, seed=SEED,
                    session_sha256={name: digest_file(SESSION_DATA / name) for name in SESSION_FILES},
                    xgb_sha256={name: digest_file(XGB_DATA / name) for name in XGB_FILES})
    if (xgb_manifest['train_hash'] != contract['session_sha256']['train_solo_track.parquet'] or
            xgb_manifest['test_hash'] != contract['session_sha256']['test_solo_track.parquet']):
        raise ValueError('Датасеты XGBoost rich и Session6 используют разные входные файлы')
    if (pd.Timestamp(XGB_PROTOCOL['val_start']) != OOF_START or
            pd.Timestamp(XGB_PROTOCOL['calibration_split']) != OOF_WINDOWS[-1][0] or
            pd.Timestamp(XGB_PROTOCOL['test_start']) != LOCAL_TEST_START or
            pd.Timestamp(xgb_evaluation['train_val_model_target_end_exclusive']) != LOCAL_TEST_START or
            XGB_PROTOCOL['model_clusters'] != [8] or XGB_PROTOCOL['test_used_for_selection'] or
            not xgb_evaluation['test_evaluated_before_full_refit']):
        raise ValueError('Границы OOF или протокол архива XGB rich отличаются')
    run_id = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:12]
    ARTIFACTS = Path('/kaggle/working') / f'best_blend_{run_id}'
    ARTIFACTS.mkdir(parents=True, exist_ok=True)
    write_json(ARTIFACTS / 'inputs.json', dict(contract, session_path=str(SESSION_DATA), xgb_path=str(XGB_DATA)))
    print('Session6:', SESSION_DATA)
    print('XGBoost rich k8:', XGB_DATA)
    history = pd.read_parquet(train_path, columns=['timestamp', 'route_id', TARGET, *STATUSES])
    history = history.sort_values(['timestamp', 'route_id'])
    TIMES = pd.DatetimeIndex(history.timestamp.unique())
    ROUTES = np.sort(history.route_id.unique())
    if (len(history) != len(TIMES) * len(ROUTES) or history.isna().any().any() or
            history.duplicated(['timestamp', 'route_id']).any() or
            not TIMES.equals(pd.date_range(TIMES[0], TIMES[-1], freq=STEP))):
        raise ValueError('Требуется полная получасовая панель')
    VALUES = {name: history[name].to_numpy(dtype=np.float32).reshape(len(TIMES), len(ROUTES))
              for name in (TARGET, *STATUSES)}
    if not all(np.isfinite(array).all() for array in VALUES.values()) or np.any(VALUES[TARGET] < 0):
        raise ValueError('Некорректные наблюдения исходной панели')
    display(pd.DataFrame([dict(split=f'oof_{i}', start=start, end_exclusive=stop)
                          for i, (start, stop) in enumerate(OOF_WINDOWS, 1)] +
                         [dict(split='test', start=LOCAL_TEST_START, end_exclusive=LOCAL_TEST_END)]))


def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))


def fit_report(model, boundary, horizon, group, split, train_y, train_pred, query_y, query_pred):
    report = pd.DataFrame([
        dict(model=model, fit_before=boundary, horizon=horizon, group=group, period=period, **metric(y, pred))
        for period, y, pred in [('train', train_y, train_pred), (split, query_y, query_pred)]
    ])
    display(report)
    path = ARTIFACTS / f'fits_{model}_{boundary:%Y%m%d}_h{horizon}_{group}.csv'
    report.to_csv(path, index=False)


def read_xgb(stage):
    if stage not in XGB_STAGES:
        raise ValueError(f'Неизвестный этап XGBoost rich: {stage}')
    frames = [pd.read_parquet(XGB_DATA / f'{stage}_h{h}_xgb_k8.parquet',
                              columns=['cutoff', 'timestamp', 'route_id', 'h', 'y', 'y_pred']) for h in HORIZONS]
    result = pd.concat(frames, ignore_index=True).rename(columns={'h': 'horizon', 'y_pred': 'xgb_rich'})
    if not result.timestamp.eq(result.cutoff + result.horizon * STEP).all():
        raise ValueError('Неверный timestamp в XGB rich')
    return result


def read_session6_test():
    frame = pd.read_parquet(SESSION_DATA / 'validation_predictions.parquet',
                            columns=KEYS + ['y', 'session_prediction'])
    return frame.rename(columns={'session_prediction': 'session6'})


def context(start, stop):
    origins = np.flatnonzero((TIMES >= start) & (TIMES < stop) & (TIMES.hour == 10) & (TIMES.minute == 30))
    expected_days = (stop - start).days
    if len(origins) != expected_days or origins[-1] + max(HORIZONS) >= len(TIMES):
        raise ValueError('Не хватает дней или целевых горизонтов')
    y = VALUES[TARGET]
    windows = np.arange(-47, 1)
    level24 = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64) for pos in origins])
    seasonal_origins = origins[:, None] - 48 * np.arange(1, 29)
    if seasonal_origins.min() + windows.min() < 0:
        raise ValueError('Не хватает наблюдений для profile28')
    seasonal_level = np.stack([y[pos + windows].mean(axis=0, dtype=np.float64)
                               for pos in seasonal_origins.ravel()]).reshape(*seasonal_origins.shape, len(ROUTES))
    ratio = level24[:, None, :] / np.maximum(seasonal_level, 1)
    route_count = len(ROUTES)
    common = dict(cutoff=TIMES[origins].repeat(route_count), route_id=np.tile(ROUTES, len(origins)),
                  level24=level24.ravel(), current=y[origins].ravel(),
                  delta1h=(y[origins] - y[origins - 2]).ravel(),
                  delta2h=(y[origins] - y[origins - 4]).ravel(),
                  network_current=np.repeat(y[origins].sum(axis=1, dtype=np.float64), route_count))
    return pd.concat([pd.DataFrame(dict(common, horizon=h, y=y[origins + h].ravel(),
                                        profile28=(y[seasonal_origins + h] * ratio).mean(axis=1).ravel()))
                      for h in HORIZONS], ignore_index=True).sort_values(KEYS).reset_index(drop=True)


def join_base(frame, incoming, column):
    relevant = incoming[KEYS + ['y', column]]
    merged = frame.merge(relevant, on=KEYS, how='left', validate='one_to_one', suffixes=('', '_incoming'))
    if len(relevant) != len(frame) or not np.array_equal(merged.y, merged.y_incoming):
        raise ValueError(f'Не совпали ключи или факты для {column}')
    if not np.isfinite(merged[column]).all() or merged[column].lt(0).any():
        raise ValueError(f'Некорректные прогнозы {column}')
    return merged.drop(columns='y_incoming')


def assemble(start, stop, session, rf, cat, xgb, xgb_clean):
    frame = context(start, stop)
    for source, name in [(session, 'session6'), (rf, 'rf'), (cat, 'cat_rich'),
                         (xgb, 'xgb_rich'), (xgb_clean, 'xgb_clean')]:
        part = source.loc[source.cutoff.ge(start) & source.cutoff.lt(stop)]
        frame = join_base(frame, part, name)
    return frame


def archive_report():
    path = ARTIFACTS.with_suffix('.zip')
    temp = path.with_suffix('.partial')
    with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=2) as archive:
        for file in sorted(ARTIFACTS.rglob('*')):
            if file.is_file() and not file.name.endswith('.partial'):
                archive.write(file, file.relative_to(ARTIFACTS.parent))
    temp.replace(path)
    print('Архив результатов', path)


In [ ]:
NOTEBOOK_HASH = '6ef620870ead8e95c70b80ea212e9ac335a29a0f2654c1059f0d699fb0b9fd1d'
import cupy as cp
from cuml.ensemble import RandomForestRegressor
from catboost.utils import get_gpu_device_count
import lightgbm as lgb
import torch

if cp.cuda.runtime.getDeviceCount() < 1 or get_gpu_device_count() < 1 or not torch.cuda.is_available():
    raise RuntimeError('Включите GPU в Kaggle и проверьте CUDA для cuML, CatBoost и PyTorch.')
SESSION_DATA_DIR = None
XGB_DATA_DIR = None
initialize(SESSION_DATA_DIR, XGB_DATA_DIR)


## Session6 и объём горизонта

На четырёх OOF-неделях используем рецепт Session6. Базовые CatBoost и LightGBM обучаются один раз до 13 сентября. Прогнозы 13–19 сентября дают первые остатки. Затем две модели остатка CatBoost обучаются заново к каждой следующей неделе на доступных более ранних прогнозах. Причинный EWMA7 учитывает прошлые факты с запасом четыре часа. Для локального test используем архивный прогноз Session6, соответствующий исходному решению 0.326. Это учитывает нужный контроль и создаёт различие между восстановленным OOF и архивным test. Его нужно иметь в виду при интерпретации переноса весов.

У всех вариантов формы в каждом старте и горизонте одна сумма. Она равна сумме Session6 по маршрутам. Отдельные варианты потом проверяют ограниченную поправку к этой сумме.


In [ ]:
session_runtime = dict(globals())
exec(compile(r'''

from catboost import CatBoostRegressor

times, routes, values = TIMES, ROUTES, VALUES
target = original_target = values[TARGET]
names = [TARGET, *STATUSES]
keys = KEYS
meta = SESSION_META
weight = float(meta['weight'])
positions = np.arange(len(times))
minutes = times.hour * 60 + times.minute
def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f'target_back_{k}'] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f'{name}_mean_{steps}'] = means[name, steps][origins]
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            if name in STATUSES[:3]:
                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)
    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)
    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)
    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])
    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка признаков не сопоставлена с историей')
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))
        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)
    return frame

def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}
    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)
    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    for days in (1, 2, 7):
        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()
    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric})

def add_pilot_features(frame, h, origins):
    current = target[origins]
    level = level24[origins]
    frame['level24_feature'] = level.ravel()
    frame['target_level24_logratio'] = (np.log1p(current) - np.log1p(level)).ravel()
    frame['mean2h_level24_logratio'] = (np.log1p(means[TARGET, 4][origins]) - np.log1p(level)).ravel()
    frame['profile28_minus_daily7'] = (frame.profile28 - frame.slot_daily7).astype(np.float32)
    return frame.astype({c: 'float32' for c in [*SEASONAL_FEATURES, *STATUS_FEATURES]})
SEASONAL_FEATURES = ['profile28', 'level24_feature', 'target_level24_logratio', 'mean2h_level24_logratio', 'profile28_minus_daily7']
STATUS_FEATURES = []

def extend_robust(data):
    new = {}
    floor = 1 / np.maximum(data.scale.to_numpy(dtype=float), 1)
    for j in range(1, 7):
        name = f'status_{j}'
        level = np.maximum(data[f'long_{name}_mean48'].to_numpy(dtype=float), floor)
        week = np.maximum(data[f'long_{name}_mean336'].to_numpy(dtype=float), floor)
        for suffix, key in [('current', f'{name}_lag0'), ('mean4', f'{name}_mean4'), ('mean12', f'{name}_mean12'), ('delta4', f'{name}_delta4')]:
            new[f'robust_s{j}_{suffix}'] = np.clip(data[key].to_numpy() / level, -100, 100).astype(np.float32)
        new[f'robust_s{j}_level_change'] = np.clip(level / week, 0, 100).astype(np.float32)
        raw = data[f'{name}_lag0'].to_numpy(dtype=float) * data.scale.to_numpy()
        denominator = pd.Series(raw).groupby([data.cutoff, data.horizon], sort=False).transform('sum').to_numpy()
        new[f'robust_s{j}_route_share'] = np.divide(raw, np.maximum(denominator, 1)).astype(np.float32)
    for window in [4, 12]:
        new[f'robust_target_vs{window}'] = np.clip(data.target_lag_0.to_numpy() / np.maximum(data[f'target_mean_{window}'].to_numpy(), floor), 0, 100).astype(np.float32)
    return pd.concat([data, pd.DataFrame(new, index=data.index)], axis=1)

def features(origins, h, profile_history, daily):
    T = len(times)
    R = len(routes)

    def past(a, index):
        index = np.asarray(index)
        out = a[np.clip(index, 0, T - 1)].astype(np.float32, copy=True)
        out[(index < 0) | (index >= T)] = np.nan
        return out
    scale = np.maximum(daily[origins], 1)
    common = {}
    for lag in (*range(9), 12, 24, 48, 96, 336):
        common[f'target_lag_{lag}'] = past(target, origins - lag) / scale
    for window in (4, 12, 48):
        common[f'target_mean_{window}'] = (daily if window == 48 else means['target_1h', window])[origins] / scale
    for window in (4, 12):
        common[f'target_std_{window}'] = deviations[window][origins] / scale
    for name in names[1:]:
        a = values[name]
        for lag in (0, 1, 2):
            common[f'{name}_lag{lag}'] = past(a, origins - lag) / scale
        for window in (4, 12):
            common[f'{name}_mean{window}'] = means[name, window][origins] / scale
        for lag in (2, 4):
            common[f'{name}_delta{lag}'] = (a[origins] - past(a, origins - lag)) / scale
    common['log_current_level'] = np.log1p(scale)
    for name, a in values.items():
        total = a.sum(axis=1, dtype=np.float64)
        netlevel = pd.Series(total).rolling(48, min_periods=1).mean().to_numpy()
        net12 = pd.Series(total).rolling(12, min_periods=1).mean().to_numpy()
        den = np.maximum(netlevel[origins], 1)
        for suffix, x in [('level', total[origins] / den), ('mean12', net12[origins] / den), ('delta2', (total[origins] - total[origins - 2]) / den)]:
            common[f'network_{name}_{suffix}'] = np.broadcast_to(x[:, None], scale.shape).astype(np.float32)
    common['route_network_share'] = target[origins] / np.maximum(target[origins].sum(axis=1, keepdims=True), 1)
    slot = origins + h
    dates = times[origins] + pd.Timedelta(minutes=30 * h)
    arrays = {name: a.ravel() for name, a in common.items()}
    for day in (1, 2, 3, 7, 14, 21, 28):
        a = past(target, slot - day * 48)
        arrays[f'slot_day{day}'] = (a / scale).ravel()
    for day in (1, 7):
        for offset in (-2, -1, 1, 2):
            arrays[f'slot_day{day}_offset{offset}'] = (past(target, slot - day * 48 + offset) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in range(1, 8)])
    arrays['slot_daily7'] = (np.nanmean(stack, axis=0) / scale).ravel()
    stack = np.stack([past(target, slot - day * 48) for day in (7, 14, 21, 28)])
    with np.errstate(invalid='ignore'):
        weekly = np.nanmean(stack, axis=0)
    arrays['slot_weekly4'] = (weekly / scale).ravel()
    profile_sum = np.zeros_like(scale, dtype=np.float64)
    count = np.zeros_like(scale)
    for day in range(1, 29):
        a = past(profile_history, slot - day * 48) * scale / np.maximum(past(daily, origins - day * 48), 1)
        valid = np.isfinite(a)
        profile_sum += np.nan_to_num(a)
        count += valid
    profile = np.divide(profile_sum, count, out=np.zeros_like(profile_sum), where=count > 0).astype(np.float32)
    arrays['profile_norm'] = (profile / scale).ravel()
    arrays['horizon'] = np.full(len(origins) * R, h, dtype=np.int8)
    arrays['dayofweek'] = np.repeat(dates.dayofweek, R).astype(np.int8)
    arrays['origin_minute'] = np.repeat(times[origins].hour * 60 + times[origins].minute, R).astype(np.int16)
    arrays['future_hour'] = np.repeat(dates.hour + dates.minute / 60, R).astype(np.float32)
    frame = pd.DataFrame(arrays)
    frame['cutoff'] = times[origins].repeat(R)
    frame['timestamp'] = dates.repeat(R)
    frame['route_id'] = np.tile(routes, len(origins)).astype(np.int16)
    frame['scale'] = scale.ravel()
    frame['profile28'] = profile.ravel()
    frame['y'] = past(target, slot).ravel()
    extra = {}
    for name in names[1:]:
        for lag in (4, 8, 12, 24, 48, 96):
            extra[f'long_{name}_lag{lag}'] = (past(values[name], origins - lag) / scale).ravel()
        for window, series in [(48, means[name, 48]), (336, long_week[name])]:
            extra[f'long_{name}_mean{window}'] = (series[origins] / scale).ravel()
    return extend_robust(pd.concat([frame, pd.DataFrame(extra)], axis=1))

def prepare_history(branch):
    global values, target, means, level24, extra_signals, hub_signal, validation_lag_table, long_week, deviations
    values = branch
    target = values[TARGET]
    means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
    level24 = rolling(target, 48)
    positions = np.arange(len(times))
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
    extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
    extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
    for name in STATUSES[3:]:
        extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
    extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
    extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
    extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
    hub_signal = extra_signals['status_6_diff_hourly']
    means.update({(name, 48): rolling(a, 48) for name, a in values.items()})
    long_week = {name: rolling(values[name], 336) for name in STATUSES}
    deviations = {w: pd.DataFrame(target).rolling(w, min_periods=2).std().to_numpy(dtype=np.float32) for w in (4, 12)}
    validation_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=start_tune)

cat_columns = ['route_id', 'target_back_0', 'target_back_1', 'target_back_2', 'target_back_4', 'target_1h_mean_4', 'target_1h_mean_12', 'status_1_mean_4', 'status_1_mean_12', 'status_1_latest', 'status_1_change_1h', 'status_2_mean_4', 'status_2_mean_12', 'status_2_latest', 'status_2_change_1h', 'status_3_mean_4', 'status_3_mean_12', 'status_3_latest', 'status_3_change_1h', 'status_4_mean_4', 'status_4_mean_12', 'status_4_latest', 'status_5_mean_4', 'status_5_mean_12', 'status_5_latest', 'status_6_mean_4', 'status_6_mean_12', 'status_6_latest', 'status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate', 'status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck', 'target_acceleration', 'target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48', 'target_vs_mean_6h', 'morning_level_ratio', 'hour', 'minute', 'hour_float', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available', 'slot_days_back_1', 'slot_days_back_2', 'slot_days_back_7', 'slot_daily7', 'slot_weekly4']
cat_trees = [1500, 1500, 1500, 1500, 1499, 1500, 1497, 1500]
anchor_columns = ['target_back_0', 'target_back_1', 'target_back_2', 'target_back_4', 'target_1h_mean_4', 'target_1h_mean_12', 'level24_feature', 'target_1h_lag_48', 'target_1h_lag_96', 'target_vs_mean_6h', 'morning_level_ratio', 'warehouse_processing_rate', 'target_acceleration']
anchor_columns += [f'status_{j}_{suffix}' for j in range(1, 7) for suffix in ('latest', 'mean_4', 'mean_12')]

def make_cat_frame(origins, h):
    return add_pilot_features(horizon_dataset(h, origins), h, origins)

def metric(actual, predicted):
    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    if y.shape != p.shape or not np.isfinite(y).all() or (not np.isfinite(p).all()) or (y.sum() <= 0):
        raise ValueError('Неверные данные для метрики')
    error = p - y
    wape = np.abs(error).sum() / y.sum()
    bias = error.sum() / y.sum()
    return dict(rows=len(y), WAPE=wape, relative_bias=bias, score=wape + abs(bias))
log = []

def report(branch, stage, h, split, y, p):
    log.append(dict(branch=branch, stage=stage, h=h, split=split, **metric(y, p)))

def show_fit():
    display(pd.DataFrame(log[-2:]))
    pd.DataFrame(log).to_csv(out / 'metrics_after_fit.csv', index=False)

def base_predictions(branch, h):
    eligible = positions[times + h * STEP < start_tune]
    cat_origins = eligible[len(eligible) // 4:]
    columns = cat_columns + (SEASONAL_FEATURES if h in (5, 6, 8) else [])
    chunks = []
    for batch in range(0, len(cat_origins), 32):
        chunks.append(make_cat_frame(cat_origins[batch:batch + 32], h)[columns])
    train = pd.concat(chunks, ignore_index=True)
    del chunks
    cat = CatBoostRegressor(iterations=cat_trees[h - 1], depth=7, learning_rate=0.05, loss_function='MAE', task_type='GPU', devices='0', cat_features=['route_id'], random_seed=42, allow_writing_files=False, verbose=250, thread_count=4)
    cat.fit(train, target[cat_origins + h].ravel() / 100000)
    model_dir = out / 'models' / branch
    cat.save_model(str(model_dir / f'catboost_h{h}.cbm'))
    fit_prediction = np.maximum(cat.predict(train) * 100000, 0)
    report(branch, 'CatBoost', h, 'train', original_target[cat_origins + h].ravel(), fit_prediction)
    del train, fit_prediction
    f = make_cat_frame(eval_origins, h)
    frame = f[['cutoff', 'route_id', 'timestamp', 'profile28', *anchor_columns]].copy()
    frame['horizon'] = h
    frame['y'] = original_target[eval_origins + h].ravel()
    frame['fit_y'] = target[eval_origins + h].ravel()
    frame['model_prediction'] = np.maximum(cat.predict(f[columns]) * 100000, 0)
    test = frame.cutoff >= start_test
    report(branch, 'CatBoost', h, 'oof', frame.loc[test, 'y'], frame.loc[test, 'model_prediction'])
    show_fit()
    frame['raw_blend'] = frame.model_prediction if h == 1 else 0.7 * frame.model_prediction + 0.3 * frame.profile28
    del cat, f
    origins = eligible[(eligible >= 336) & np.isin(minutes[eligible], [570, 600, 630, 660, 690])]
    train = features(origins, h, target, level24)
    cfg = meta[str(h)]
    cols = cfg['columns']
    model = lgb.LGBMRegressor(**dict(cfg['params'], n_estimators=cfg['trees'], n_jobs=4))
    scale = train.scale.to_numpy(dtype=float)
    model.fit(train[cols], target[origins + h].ravel() / scale, sample_weight=scale / scale.mean())
    model.booster_.save_model(str(model_dir / f'lightgbm_h{h}.txt'))
    fit_prediction = np.maximum(model.predict(train[cols]) * scale, 0)
    report(branch, 'LightGBM', h, 'train', original_target[origins + h].ravel(), fit_prediction)
    del train, fit_prediction
    evaluation = features(eval_origins, h, target, level24)
    if not frame[keys].reset_index(drop=True).equals(evaluation[keys].astype(frame[keys].dtypes).reset_index(drop=True)):
        raise ValueError('Расходятся ключи между базовыми моделями')
    frame['shape'] = np.maximum(model.predict(evaluation[cols]) * evaluation.scale.to_numpy(dtype=float), 0)
    report(branch, 'LightGBM', h, 'oof', frame.loc[test, 'y'], frame.loc[test, 'shape'])
    show_fit()
    return frame

def residual_features(frame):
    cols = ['route_id', 'horizon', 'raw_blend', 'model_prediction', 'profile28', *anchor_columns]
    X = frame[cols].copy()
    X['dayofweek'] = frame.cutoff.dt.dayofweek
    X['route_id'] = X.route_id.astype(str)
    X['horizon'] = X.horizon.astype(str)
    for name in ['model_prediction', 'profile28', 'target_back_0', 'target_1h_mean_4', 'target_1h_mean_12', 'level24_feature', 'target_1h_lag_48', 'target_1h_lag_96']:
        X[f'{name}_minus_base'] = X[name] - X.raw_blend
    return X

def causal_offsets(frame, initial, unshifted, fit_mask):
    errors = frame.fit_y.to_numpy() - unshifted
    offsets = np.full(len(frame), initial)
    for cutoff in frame.loc[~fit_mask, 'cutoff'].drop_duplicates():
        known = frame.timestamp < cutoff - pd.Timedelta(hours=4)
        if frame.loc[known, 'cutoff'].nunique() >= 3:
            ages = (cutoff - frame.loc[known, 'cutoff']).dt.total_seconds().to_numpy() / 86400
            offsets[frame.cutoff.eq(cutoff)] = np.average(errors[known], weights=np.exp2(-ages / 7))
    return offsets

def finish_pipeline(frame, branch, fit_before):
    fit = frame.cutoff < fit_before
    test = frame.cutoff >= start_test
    X = residual_features(frame)
    corrected = []
    for depth, trees, l2 in [(4, 450, 30), (5, 900, 50)]:
        model = CatBoostRegressor(iterations=trees, depth=depth, learning_rate=0.03, l2_leaf_reg=l2, loss_function='MAE', cat_features=['route_id', 'horizon'], random_seed=42, thread_count=4, allow_writing_files=False, verbose=False)
        model.fit(X.loc[fit], (frame.loc[fit, 'fit_y'] - frame.loc[fit, 'raw_blend']) / 100000)
        unshifted = frame.raw_blend.to_numpy() + model.predict(X) * 100000
        initial = np.mean(frame.loc[fit, 'fit_y'] - unshifted[fit])
        folder = out / 'models' / branch / f'corrections_before_{fit_before:%Y%m%d}'
        folder.mkdir(parents=True, exist_ok=True)
        model.save_model(str(folder / f'depth{depth}.cbm'))
        state = frame[keys + ['timestamp']].copy()
        state['residual'] = frame.fit_y.to_numpy() - unshifted
        state['initial_offset'] = float(initial)
        state['used_for_fit'] = fit.to_numpy()
        state.to_parquet(folder / f'depth{depth}_history.parquet', index=False)
        pred = np.maximum(unshifted + causal_offsets(frame, initial, unshifted, fit), 0)
        stage = f'residual_depth{depth}_fit_before_{fit_before:%m%d}'
        report(branch, stage, 'all', 'train', frame.loc[fit, 'y'], pred[fit])
        report(branch, stage, 'all', 'oof', frame.loc[test, 'y'], pred[test])
        show_fit()
        corrected.append(pred)
    p = frame[keys + ['y', 'shape']].copy()
    p['volume_model'] = (corrected[0] + corrected[1]) / 2
    totals = p.groupby(['cutoff', 'horizon'])[['volume_model', 'shape']].transform('sum')
    if totals['shape'].le(0).any():
        raise ValueError('Нулевая сумма LightGBM')
    p['prediction'] = (1 - weight) * p.volume_model + weight * p['shape'] * totals.volume_model / totals['shape']
    p['branch'] = branch
    return p

def session6_oof():
    global start_tune, start_test, eval_origins, out
    out = ARTIFACTS / 'session6'
    (out / 'models' / 'early_anchor').mkdir(parents=True, exist_ok=True)
    path = out / 'oof.parquet'
    if path.is_file():
        return pd.read_parquet(path)
    start_tune = OOF_START - pd.Timedelta(days=7)
    eval_origins = positions[(times >= start_tune) & (times < LOCAL_TEST_START) & (minutes == 630)]
    start_test = start_tune
    prepare_history(values)
    base = pd.concat([base_predictions('early_anchor', h) for h in HORIZONS], ignore_index=True)
    base = base.sort_values(keys).reset_index(drop=True)
    blocks = []
    for start, stop in OOF_WINDOWS:
        start_test = start
        block = base.loc[base.cutoff.lt(stop)].reset_index(drop=True)
        prediction = finish_pipeline(block, 'early_anchor', start)
        result = prediction.loc[prediction.cutoff.ge(start), keys + ['y', 'prediction', 'volume_model', 'shape']]
        blocks.append(result.rename(columns={'prediction': 'session6'}))
    result = pd.concat(blocks, ignore_index=True)
    save_parquet(path, result)
    return result
''', 'session_runtime', 'exec'), session_runtime)


## CatBoost rich с восемью кластерами

Перед каждой OOF-неделей заново считаются статистики маршрутов, KMeans и выбранные динамические лаги. Модели CatBoost MAE обучаются по кластерам на всех получасовых стартах и всех целях до границы окна. Для слишком малых кластеров используется общая модель. Признаки, параметры и число деревьев взяты из rich-ноутбука. Прогнозы сохраняются без калибровки.


In [ ]:
cat_runtime = dict(globals())
exec(compile(r'''

import joblib
from catboost import CatBoostRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

CLUSTERS = (1, 8)
MIN_CLUSTER_ROUTES = 10
RUN = ARTIFACTS / 'cat_rich'
RUN.mkdir(exist_ok=True)
ROLL = {w: pd.DataFrame(VALUES[TARGET]).rolling(w, min_periods=1).mean().to_numpy(dtype=np.float32)
        for w in (4, 12, 48, 336)}


PARAMETERS = {'selected_cat': {'1': {'params': {'depth': 6, 'learning_rate': 0.03694755509074355, 'l2_leaf_reg': 1.5679933916723008, 'random_strength': 2.052699079536471, 'bagging_temperature': 0.8803049874792026}, 'trees': 900}, '2': {'params': {'depth': 5, 'learning_rate': 0.1358198535217987, 'l2_leaf_reg': 29.10635913133069, 'random_strength': 1.7959754525911098, 'bagging_temperature': 0.31203728088487304}, 'trees': 387}, '3': {'params': {'depth': 4, 'learning_rate': 0.0224831270535195, 'l2_leaf_reg': 53.99484409787433, 'random_strength': 1.8033450352296265, 'bagging_temperature': 1.416145155592091}, 'trees': 900}, '4': {'params': {'depth': 6, 'learning_rate': 0.0299058938693663, 'l2_leaf_reg': 10.677482709481351, 'random_strength': 1.7772437065861273, 'bagging_temperature': 0.09290082543999545}, 'trees': 364}, '5': {'params': {'depth': 4, 'learning_rate': 0.0224831270535195, 'l2_leaf_reg': 53.99484409787433, 'random_strength': 1.8033450352296265, 'bagging_temperature': 1.416145155592091}, 'trees': 504}, '6': {'params': {'depth': 4, 'learning_rate': 0.1411758906434134, 'l2_leaf_reg': 46.22589001020831, 'random_strength': 0.6370173320348285, 'bagging_temperature': 0.36364993441420124}, 'trees': 125}, '7': {'params': {'depth': 4, 'learning_rate': 0.1411758906434134, 'l2_leaf_reg': 46.22589001020831, 'random_strength': 0.6370173320348285, 'bagging_temperature': 0.36364993441420124}, 'trees': 167}, '8': {'params': {'depth': 4, 'learning_rate': 0.03691990689655602, 'l2_leaf_reg': 11.207606211860567, 'random_strength': 1.2958350559263474, 'bagging_temperature': 0.5824582803960838}, 'trees': 900}}}

RICH_SOURCE = "import pyarrow as pa\nimport pyarrow.parquet as pq\ntarget = values[TARGET]\n\ndef past(array, positions):\n    positions = np.asarray(positions, dtype=np.int64)\n    valid = (positions >= 0) & (positions < len(array))\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\n    result[~valid] = np.nan\n    return result\n\ndef average_available(matrices):\n    total = np.zeros_like(matrices[0], dtype=np.float32)\n    count = np.zeros_like(total)\n    for matrix in matrices:\n        valid = np.isfinite(matrix)\n        total += np.where(valid, matrix, 0)\n        count += valid\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\n\ndef rolling(array, steps):\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\nlevel24 = rolling(target, 48)\npositions = np.arange(len(times))\nextra_signals = {}\nfor name in STATUSES[:3]:\n    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)\nextra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))\nextra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))\nfor name in STATUSES[3:]:\n    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)\nextra_signals['hub_buffer'] = values['status_5'] - values['status_4']\nextra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']\nextra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)\nhub_signal = extra_signals['status_6_diff_hourly']\n\ndef common_features(origins):\n    common = {}\n    for k in (0, 1, 2, 4):\n        common[f'target_back_{k}'] = past(target, origins - k)\n    for name, array in values.items():\n        for steps in (4, 12):\n            common[f'{name}_mean_{steps}'] = means[name, steps][origins]\n        if name in STATUSES:\n            common[f'{name}_latest'] = array[origins]\n            if name in STATUSES[:3]:\n                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)\n    for name, array in extra_signals.items():\n        common[name] = array[origins]\n    for lag in (48, 96):\n        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)\n    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)\n    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)\n    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]\n    recent = means[TARGET, 4][origins]\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\n    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\n    return common\nCORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']\nFEATURE_GROUPS = {'calendar': ['hour', 'minute', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': ['status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'], 'hub_combinations': ['status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'], 'historical_lags': ['target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48'], 'target_change': ['target_acceleration'], 'correlation_lag': CORRELATION_COLUMNS, 'pilot_start_time': ['start_sin', 'start_cos'], 'pilot_peaks': ['peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']}\n\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\n    grid = pd.DatetimeIndex(timestamps)\n    route_ids = np.asarray(route_ids)\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\n        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\n        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')\n    if len(np.unique(route_ids)) != len(route_ids):\n        raise ValueError('Идентификаторы маршрутов повторяются')\n    steps = float(max_lookback_hours) * 2\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\n        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\n        raise ValueError('min_pairs должен быть целым числом >=3')\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\n        raise ValueError('Некорректные параметры выбора корреляции')\n    max_lag = int(steps)\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))\n    anchors = np.arange(max_lag + 2, stop)\n    best_corr = np.full(len(route_ids), np.nan)\n    best_lag = np.full(len(route_ids), np.nan)\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\n    for start in range(0, len(route_ids), route_block_size):\n        finish = min(start + route_block_size, len(route_ids))\n        x = target_values[anchors, start:finish].astype(np.float64)\n        for lag in range(2, max_lag + 1):\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\n            valid = np.isfinite(x) & np.isfinite(z)\n            count = valid.sum(axis=0)\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\n            divisor = np.maximum(count, 1)\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\n            denominator = np.sqrt(vx * vz)\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\n            corr = np.clip(corr, -1, 1)\n            previous = best_corr[start:finish]\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\n            best_corr[start:finish][better] = corr[better]\n            best_lag[start:finish][better] = lag\n            best_pairs[start:finish][better] = count[better]\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\n    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})\n\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\n    if h not in range(1, 9) or chunk_rows < 1:\n        raise ValueError('Некорректный горизонт или размер блока')\n    by_route = lag_table.set_index('route_id').reindex(routes)\n    route_index = pd.Index(routes)\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\n    for name in CORRELATION_COLUMNS:\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\n    for start in range(0, len(frame), chunk_rows):\n        finish = min(start + chunk_rows, len(frame))\n        block = frame.iloc[start:finish]\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\n        route = route_index.get_indexer(block.route_id)\n        if (origin < 0).any() or (route < 0).any():\n            raise ValueError('Строка признаков не сопоставлена с историей')\n        selected = lag[route]\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\n        available &= np.isfinite(signal)\n        for name, array in quality.items():\n            frame.iloc[start:finish, column_positions[name]] = array[route]\n        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)\n        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)\n    return frame\n\ndef horizon_dataset(h, origins):\n    slot = origins + h\n    dates = times[origins] + h * STEP\n    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})\n    for name, matrix in common_features(origins).items():\n        frame[name] = matrix.ravel()\n    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}\n    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)\n    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)\n    for name, array in calendar.items():\n        frame[name] = np.repeat(array, len(routes))\n    frame = add_correlation_features(frame, h, validation_lag_table)\n    for days in (1, 2, 7):\n        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()\n    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\n    profile = np.zeros((len(origins), len(routes)))\n    counts = np.zeros_like(profile)\n    for day in range(1, 29):\n        level = past(level24, origins - day * 48)\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\n        profile += np.nan_to_num(adjusted)\n        counts += np.isfinite(adjusted)\n    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\n    if h == 1:\n        lower = np.zeros_like(target[origins])\n        upper = past(target, origins - 2)\n        for position in (origins - 1, origins):\n            total = past(target, position)\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\n        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()\n        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\n    else:\n        frame['overlap'] = np.nan\n    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])\n    return frame.astype({name: 'float32' for name in numeric})"

def segment_routes(boundary, stage):
    mask = TIMES < boundary
    y = VALUES[TARGET][mask].astype(np.float64)
    scale = np.maximum(y.mean(axis=0), 1.0)
    normalized = y / scale
    grid = TIMES[mask]
    descriptors = {}
    for slot in range(48):
        select = grid.hour * 2 + grid.minute // 30 == slot
        descriptors[f'slot_{slot:02}'] = normalized[select].mean(axis=0)
    for day in range(7):
        descriptors[f'weekday_{day}'] = normalized[grid.dayofweek == day].mean(axis=0)
    descriptors['log_cv'] = np.log1p(normalized.std(axis=0))
    descriptors['zero_share'] = (y == 0).mean(axis=0)
    descriptors['jump_share'] = (np.abs(np.diff(normalized, axis=0)) > 1).mean(axis=0)
    descriptors['log_q95'] = np.log1p(np.quantile(normalized, 0.95, axis=0))
    frame = pd.DataFrame(descriptors, index=ROUTES)
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError('Недостаточно данных для профилей маршрутов')
    scaler = StandardScaler().fit(frame)
    z = scaler.transform(frame)
    labels, estimators = ({1: np.zeros(len(ROUTES), dtype=int)}, {})
    for k in CLUSTERS:
        if k == 1:
            continue
        model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(z)
        groups = model.labels_.copy()
        counts = np.bincount(groups, minlength=k)
        groups[counts[groups] < MIN_CLUSTER_ROUTES] = -1
        labels[k], estimators[k] = (groups, model)
    artifact = dict(boundary=str(boundary), routes=ROUTES, scale=scale, labels=labels, descriptor_scaler=scaler, clusterers=estimators, descriptor_columns=list(frame.columns))
    joblib.dump(artifact, RUN / f'segments_{stage}.joblib')
    frame.insert(0, 'route_id', ROUTES)
    frame['target_scale'] = scale
    for k, groups in labels.items():
        frame[f'cluster_{k}'] = groups
    frame.to_csv(RUN / f'route_profiles_{stage}.csv', index=False)
    display(pd.DataFrame([dict(stage=stage, k=k, cluster=int(g), routes=int((groups == g).sum())) for k, groups in labels.items() for g in np.unique(groups)]))
    return artifact

def features(h, origins, segmentation):
    scale = segmentation['scale'].astype(np.float32)
    columns, names = ([], [])
    shape = (len(origins), len(ROUTES))

    def add(name, array):
        names.append(name)
        columns.append(np.broadcast_to(array, shape).reshape(-1))

    def lag(name, array, indices):
        valid = indices >= 0
        data = array[np.maximum(indices, 0)] / scale
        add(name, np.where(valid[:, None], data, 0))
        add(name + '_available', valid[:, None])
    for delay in (0, 1, 2, 4, 12):
        lag(f'target_back_{delay}', VALUES[TARGET], origins - delay)
    for status in STATUSES:
        lag(status, VALUES[status], origins)
        add(status + '_delta_1h', (VALUES[status][origins] - VALUES[status][np.maximum(origins - 2, 0)]) / scale)
    for window, array in ROLL.items():
        lag(f'target_mean_{window}', array, origins)
    for days in (1, 7, 14, 28):
        lag(f'target_season_{days}', VALUES[TARGET], origins + h - 48 * days)
    dates = TIMES[origins] + h * STEP
    for label, phase in [('weekday', dates.dayofweek.to_numpy() / 7), ('slot', (dates.hour * 2 + dates.minute / 30).to_numpy() / 48)]:
        add(label + '_sin', np.sin(2 * np.pi * phase)[:, None])
        add(label + '_cos', np.cos(2 * np.pi * phase)[:, None])
    add('log_route_scale', np.log1p(scale)[None, :])
    matrix = np.column_stack(columns).astype(np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError('Неконечные признаки')
    return (matrix, names)

original_features_rich = features
original_segment_rich = segment_routes

def segment_routes(boundary, stage):
    global FEATURE_NS
    result = original_segment_rich(boundary, stage)
    FEATURE_NS = dict(np=np, pd=pd, values=VALUES, TARGET=TARGET, STATUSES=STATUSES, times=TIMES, routes=ROUTES, STEP=STEP, WINDOW_OFFSETS=tuple(range(-4, 5)))
    exec(RICH_SOURCE, FEATURE_NS)
    lag_table = FEATURE_NS['calculate_dynamic_shoulder'](VALUES[TARGET], FEATURE_NS['hub_signal'], TIMES, ROUTES, fit_before=boundary)
    FEATURE_NS['validation_lag_table'] = lag_table
    FEATURE_NS['network_values'] = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32) for name, array in VALUES.items()}
    lag_table.to_csv(RUN / f'rich_status6_lags_{stage}.csv', index=False)
    return result

def features(h, origins, segmentation):
    base, names = original_features_rich(h, origins, segmentation)
    frame = FEATURE_NS['horizon_dataset'](h, origins)
    cols = [c for c in frame if c not in ('route_id', 'cutoff', 'timestamp', 'y', 'id', 'overlap')]
    values = frame[cols].to_numpy(dtype=np.float32)
    scale = np.tile(segmentation['scale'], len(origins)).astype(np.float32)
    dimensional = [i for i, c in enumerate(cols) if c not in ('hour', 'minute', 'hour_float', 'dayofweek', 'is_weekend') and (not any((v in c for v in ('ratio', 'rate', 'correlation', 'pair_count', 'lag_steps', 'valid', 'available', '_sin', '_cos'))))]
    for i in dimensional:
        values[:, i] /= scale
    available = np.isfinite(values).astype(np.float32)
    values = np.where(np.isfinite(values), values, 0)
    shoulder_columns, shoulder_names = ([], [])
    for label, days in [('daily7', range(1, 8)), ('weekly4', (7, 14, 21, 28))]:
        profiles = []
        for offset in (-4, -2, 0, 2, 4):
            total = np.zeros((len(origins), len(ROUTES)), dtype=np.float32)
            count = np.zeros((len(origins), 1), dtype=np.float32)
            for day in days:
                positions = origins + h - day * 48 + offset
                valid = positions >= 0
                total += np.where(valid[:, None], VALUES[TARGET][np.maximum(positions, 0)], 0)
                count += valid[:, None]
            profile = np.divide(total, count, out=np.zeros_like(total), where=count > 0) / segmentation['scale'][None, :]
            profiles.append(profile.ravel())
            shoulder_columns.append(profile.ravel())
            shoulder_names.append(f'rich_{label}_shoulder_{offset}')
        window = np.column_stack(profiles)
        shoulder_columns.extend([window.max(axis=1) - window.min(axis=1), np.argmax(window, axis=1) * 60 - 120])
        shoulder_names.extend([f'rich_{label}_shoulder_range', f'rich_{label}_peak_minutes'])
    for signal, array in FEATURE_NS['network_values'].items():
        for lag in (0, 2):
            signal_values = np.where(origins >= lag, array[np.maximum(origins - lag, 0)], 0) / max(float(segmentation['scale'].sum()), 1.0)
            shoulder_columns.append(np.repeat(signal_values, len(ROUTES)))
            shoulder_names.append(f'rich_network_{signal}_back_{lag}')
        change = (array[origins] - array[np.maximum(origins - 2, 0)]) / max(float(segmentation['scale'].sum()), 1.0)
        shoulder_columns.append(np.repeat(change, len(ROUTES)))
        shoulder_names.append(f'rich_network_{signal}_change_1h')
    shoulder_columns.append(np.tile(np.arange(len(ROUTES), dtype=np.float32), len(origins)))
    shoulder_names.append('rich_route_index')
    share = VALUES[TARGET][origins] / np.maximum(FEATURE_NS['network_values'][TARGET][origins, None], 1)
    shoulder_columns.append(share.ravel())
    shoulder_names.append('rich_route_target_network_share')
    matrix = np.column_stack([base, values, available, *[np.asarray(c, dtype=np.float32) for c in shoulder_columns]]).astype(np.float32, copy=False)
    names = names + ['rich_' + c for c in cols] + ['rich_available__' + c for c in cols] + shoulder_names
    if not np.isfinite(matrix).all():
        raise ValueError('Неконечные расширенные признаки')
    return (matrix, names)

def model_frame(x, names, algorithm):
    frame = pd.DataFrame(x, columns=names)
    route_codes = x[:, names.index('rich_route_index')].astype(np.int64)
    frame['route_code'] = pd.Categorical(route_codes, categories=np.arange(len(ROUTES))) if algorithm == 'xgb' else route_codes.astype(str)
    return frame

def fit_rich_cat(start, stop, split):
    if start >= LOCAL_TEST_START and (not (ARTIFACTS / 'frozen_selection.json').is_file()):
        raise ValueError('Сначала зафиксируйте выбор по val')
    folder = RUN / start.strftime('%Y%m%d')
    folder.mkdir(parents=True, exist_ok=True)
    outputs = [folder / f'h{h}.parquet' for h in HORIZONS]
    if not all((path.is_file() for path in outputs)):
        with threadpool_limits(limits=THREADS):
            segmentation = segment_routes(start, start.strftime('%Y%m%d'))
        route_groups = segmentation['labels'][8]
        route_scale = segmentation['scale']
        origins = np.arange(len(TIMES) - 8)
        for h, path in zip(HORIZONS, outputs):
            if path.is_file():
                continue
            train_origins = np.flatnonzero(TIMES[:len(TIMES) - h] + h * STEP < start)
            query_origins = origins[(TIMES[origins] >= start) & (TIMES[origins] < stop) & (TIMES[origins].hour == 10) & (TIMES[origins].minute == 30)]
            x, columns = rich_matrix(h, train_origins, segmentation)
            query, query_columns = rich_matrix(h, query_origins, segmentation)
            if columns != query_columns:
                raise ValueError('Расходятся колонки CatBoost rich')
            y = VALUES[TARGET][train_origins + h]
            actual = VALUES[TARGET][query_origins + h]
            predictions = np.empty_like(actual, dtype=np.float64)
            spec = PARAMETERS['selected_cat'][str(h)]
            for group in np.unique(route_groups):
                query_routes = np.flatnonzero(route_groups == group)
                train_routes = np.arange(len(ROUTES)) if group == -1 else query_routes
                train_x = model_frame(x[:, train_routes].reshape(-1, len(columns)), columns, 'cat')
                train_y = y[:, train_routes].reshape(-1)
                train_scale = np.tile(route_scale[train_routes], len(train_origins))
                query_x = model_frame(query[:, query_routes].reshape(-1, len(columns)), columns, 'cat')
                model = CatBoostRegressor(**spec['params'], iterations=spec['trees'], loss_function='MAE', task_type='GPU', devices='0', thread_count=THREADS, random_seed=SEED, verbose=False, allow_writing_files=False, allow_const_label=True, bootstrap_type='Bayesian', cat_features=['route_code'])
                model.fit(train_x, train_y / train_scale, sample_weight=train_scale / route_scale.mean())
                train_pred = np.maximum(model.predict(train_x) * train_scale, 0)
                query_pred = np.maximum(model.predict(query_x).reshape(len(query_origins), -1) * route_scale[query_routes], 0)
                predictions[:, query_routes] = query_pred
                fit_report('cat_rich', start, h, int(group), split, train_y, train_pred, actual[:, query_routes], query_pred)
                del model, train_x, query_x, train_pred
                gc.collect()
            frame = pd.DataFrame(dict(cutoff=TIMES[query_origins].repeat(len(ROUTES)), route_id=np.tile(ROUTES, len(query_origins)), horizon=h, y=actual.ravel(), cat_rich=predictions.ravel()))
            save_parquet(path, frame)
            del x, query
            gc.collect()
    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)

def rich_matrix(h, origins, segmentation):
    if not len(origins):
        raise ValueError('Пустой диапазон CatBoost rich')
    routes_count = len(ROUTES)
    matrix = None
    for start in range(0, len(origins), 32):
        batch = origins[start:start + 32]
        raw, names = features(h, batch, segmentation)
        if matrix is None:
            columns = names
            matrix = np.empty((len(origins), routes_count, len(columns)), dtype=np.float32)
        if names != columns:
            raise ValueError('Расходятся колонки батчей CatBoost rich')
        matrix[start:start + len(batch)] = raw.reshape(len(batch), routes_count, len(columns))
    return (matrix, columns)
''', 'cat_runtime', 'exec'), cat_runtime)


## Random Forest с индивидуальными признаками по горизонтам

Воспроизводим сохранённый рецепт леса. Для h1–h8 остаются свои параметры и наборы признаков. Медиана по маршруту применяется только там, где она выбрана в исходной конфигурации. Лаги и заполнение пропусков оцениваются внутри обучающего периода каждой недели. Обучение использует пять утренних стартов, проверка использует 10:30.


In [ ]:
rf_runtime = dict(globals())
exec(compile(r'''

import re
import cupy as cp
from cuml.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

times, routes, values = TIMES, ROUTES, VALUES
RUN = ARTIFACTS / 'rf'
RUN.mkdir(exist_ok=True)


BASE_PARAMS_BY_HORIZON = {1: {'n_estimators': 150, 'max_depth': 16, 'min_samples_leaf': 44, 'max_features': 0.9154095569322449, 'max_samples': 0.7361074625809747}, 2: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 37, 'max_features': 0.6600327095074597, 'max_samples': 0.5924272277627636}, 3: {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 37, 'max_features': 0.6600327095074597, 'max_samples': 0.5924272277627636}, 4: {'n_estimators': 150, 'max_depth': 12, 'min_samples_leaf': 37, 'max_features': 0.6943109266465318, 'max_samples': 0.5232252063599989}, 5: {'n_estimators': 150, 'max_depth': 16, 'min_samples_leaf': 44, 'max_features': 0.9154095569322449, 'max_samples': 0.7361074625809747}, 6: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 23, 'max_features': 0.5045012539746527, 'max_samples': 0.6831809216468459}, 7: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}, 8: {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457}}
N_STREAMS = 4
PREDICTION_BATCH = 32768
TRAIN_MINUTES = (570, 600, 630, 660, 690)
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
RESIDUAL = False

best_params = BASE_PARAMS_BY_HORIZON
selected = pd.DataFrame(json.loads('[{"horizon":2,"trimmed":false,"route_median":true,"experiment":"thin_offsets","mean":0.3472783742,"std":0.0237257355,"max":0.3845543627},{"horizon":6,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":0.3505317232,"std":0.0160416261,"max":0.3713695576},{"horizon":1,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":null,"std":null,"max":null},{"horizon":3,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":4,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":5,"trimmed":false,"route_median":false,"experiment":"compact_with_additions","mean":null,"std":null,"max":null},{"horizon":7,"trimmed":false,"route_median":false,"experiment":"baseline","mean":null,"std":null,"max":null},{"horizon":8,"trimmed":false,"route_median":false,"experiment":"thin_offsets","mean":null,"std":null,"max":null}]')).set_index('horizon')

import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']
for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions - 2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)

def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    if lag_table is None:
        names = [prefix + name for name in names]
        return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)

def remove_seasonality(series):
    typical = average_available([past(series, positions - 48 * d) for d in range(1, 8)])
    return series - typical
residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target, hub_signal, ''), (residual_target, residual_signal, 'residual_')):
        table = calculate_dynamic_shoulder(y, signal, times, routes, fit_before=boundary, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c: prefix + c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1], on='route_id', validate='one_to_one')

def add_correlation_features(frame, h, lag_table):
    frame = correlation_block(frame, h, lag_table, hub_signal, '')
    if lag_table is None:
        return correlation_block(frame, h, None, residual_signal, 'residual_')
    residual_table = lag_table[['route_id', *['residual_' + c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id', *CORRELATION_COLUMNS[:4]]
    return correlation_block(frame, h, residual_table, residual_signal, 'residual_')

def stable_ratio(numerator, denominator):
    return numerator / np.maximum(np.abs(denominator), 1.0)

def available_statistics(window):
    valid = np.isfinite(window)
    count = valid.sum(axis=0)
    mean = np.divide(np.where(valid, window, 0).sum(axis=0, dtype=np.float64), count, out=np.full(window.shape[1:], np.nan), where=count > 0)
    variance = np.divide(np.where(valid, (window - mean) ** 2, 0).sum(axis=0), count, out=np.full_like(mean, np.nan), where=count > 0)
    spread = np.where(count > 0, np.where(valid, window, -np.inf).max(axis=0) - np.where(valid, window, np.inf).min(axis=0), np.nan)
    return (mean, np.sqrt(variance), spread, count)

def add_experimental_features(frame, h, origins):
    additions = {}
    additions['x_time_cutoff_minutes'] = np.repeat(times[origins].hour * 60 + times[origins].minute, len(routes))
    additions['x_profile_profile28'] = frame.profile28.to_numpy()
    additions['x_level_level24'] = level24[origins].ravel()
    typical = average_available([past(target, origins - day * 48) for day in range(1, 8)])
    current = target[origins]
    additions['x_anomaly_target_difference'] = (current - typical).ravel()
    additions['x_anomaly_target_ratio'] = stable_ratio(current, typical).ravel()
    for name in (TARGET, *STATUSES[:3]):
        short, long = (means[name, 4][origins], means[name, 12][origins])
        additions[f'x_trend_{name}_difference'] = (short - long).ravel()
        additions[f'x_trend_{name}_ratio'] = stable_ratio(short, long).ravel()
        for steps in (4, 12):
            window = np.stack([past(values[name], origins - lag) for lag in range(steps)])
            mean, std, spread, count = available_statistics(window)
            change = available_statistics(np.abs(np.diff(window, axis=0)))[0]
            additions[f'x_volatility_{name}_std_{steps}'] = std.ravel()
            additions[f'x_volatility_{name}_range_{steps}'] = spread.ravel()
            additions[f'x_volatility_{name}_change_{steps}'] = change.ravel()
    for label, days in [('daily7', range(1, 8)), ('weekly4', (7, 14, 21, 28))]:
        window = np.stack([past(target, origins + h - day * 48) for day in days])
        mean, std, spread, count = available_statistics(window)
        additions[f'x_dispersion_{label}_std'] = std.ravel()
        additions[f'x_dispersion_{label}_relative_std'] = stable_ratio(std, mean).ravel()
        additions[f'x_dispersion_{label}_count'] = count.ravel()
    return pd.concat([frame, pd.DataFrame(additions, index=frame.index)], axis=1)

def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, None)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7', 'slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base, k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:, None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window), np.nan), where=count > 0)
        seasonal_columns[base + '_peak_minutes'] = shift * 30
        seasonal_columns[base + '_range'] = np.where(count > 0, maximum - minimum, np.nan)
        seasonal_columns[base + '_right_minus_left'] = window[:, 5:].mean(axis=1) - window[:, :4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    frame = add_experimental_features(frame, h, origins)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()

EXPERIMENTS = {'baseline': dict(), 'drop_constant': dict(drop_constant=True), 'compact_availability': dict(compact_availability=True), 'calendar_dayofweek': dict(calendar=True), 'thin_offsets': dict(thin=True), 'drop_network': dict(network='drop'), 'thin_network': dict(network='thin'), 'add_time': dict(add=['time']), 'add_profile': dict(add=['profile']), 'add_level24': dict(add=['level']), 'add_anomaly': dict(add=['anomaly']), 'add_trends': dict(add=['trend']), 'add_volatility': dict(add=['volatility']), 'add_seasonal_dispersion': dict(add=['dispersion']), 'all_additions': dict(add=['time', 'profile', 'level', 'anomaly', 'trend', 'volatility', 'dispersion']), 'compact': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin'), 'compact_with_additions': dict(drop_constant=True, compact_availability=True, calendar=True, thin=True, network='thin', add=['time', 'profile', 'level', 'anomaly', 'trend', 'volatility', 'dispersion'])}

def model_target(frame):
    baseline = frame.profile28.to_numpy() if RESIDUAL else 0
    return ((frame.y.to_numpy() - baseline) / TARGET_SCALE).astype(np.float32)

def transformed_frame(frame, columns):
    result = frame[columns].drop(columns='route_id').astype(np.float32)
    codes = pd.Index(routes).get_indexer(frame.route_id).astype(np.int64) + 1
    bits = {f'route_bit_{i}': (codes >> i & 1).astype(np.float32) for i in range(max(1, int(len(routes)).bit_length()))}
    return pd.concat([result, pd.DataFrame(bits, index=frame.index)], axis=1).copy()

def fit_forest(frame, matrix, params):
    model = RandomForestRegressor(random_state=SEED, n_streams=N_STREAMS, output_type='numpy', **params)
    model.fit(matrix, model_target(frame))
    cp.cuda.get_current_stream().synchronize()
    return model

def predict_matrix(model, frame, matrix):
    result = np.empty(len(frame), dtype=np.float64)
    for start in range(0, len(frame), PREDICTION_BATCH):
        stop = min(start + PREDICTION_BATCH, len(frame))
        raw = np.asarray(model.predict(matrix[start:stop]), dtype=np.float64).reshape(-1)
        baseline = frame.profile28.iloc[start:stop].to_numpy() if RESIDUAL else 0
        result[start:stop] = np.maximum(0, raw * TARGET_SCALE + baseline)
    if not np.isfinite(result).all():
        raise ValueError('Некорректный прогноз')
    return result

def experiment_columns(names, spec):
    result = []
    for name in names:
        if name.startswith('x_'):
            if any((name.startswith(f'x_{group}_') for group in spec.get('add', []))):
                result.append(name)
            continue
        if spec.get('drop_constant') and name == 'selected_status6_pair_count':
            continue
        if spec.get('compact_availability') and 'signal_available' in name:
            continue
        if spec.get('calendar') and name in ('weekday_sin', 'weekday_cos', 'is_weekend'):
            continue
        if spec.get('network') == 'drop' and name.startswith('network_'):
            continue
        if spec.get('network') == 'thin' and name.startswith('network_') and ('_offset_' in name):
            continue
        if spec.get('thin') and (not name.startswith('slot_')) and re.search('_offset_m[134]$', name):
            continue
        result.append(name)
    if spec.get('compact_availability'):
        for name in ('status6_lag_valid', 'residual_status6_lag_valid'):
            if name in names and name not in result:
                result.append(name)
    return result

def rf_data(h, origins, lag_table, spec):
    batches, parts = ([], [])
    for start in range(0, len(origins), 24):
        batch = origins[start:start + 24]
        frame = horizon_dataset(h, batch)
        frame['y'] = target[batch + h].ravel()
        frame = frame.loc[np.isfinite(frame.profile28) & np.isfinite(frame.y)].reset_index(drop=True)
        frame = add_correlation_features(frame, h, lag_table)
        matrix = transformed_frame(frame, model_columns(frame))
        columns = experiment_columns(list(matrix.columns), EXPERIMENTS[spec.experiment])
        batches.append(matrix[columns].replace([np.inf, -np.inf], np.nan))
        parts.append(frame[['cutoff', 'timestamp', 'route_id', 'y', 'profile28']].copy())
    return (pd.concat(parts, ignore_index=True), pd.concat(batches, ignore_index=True))

def fit_rf(start, stop, split):
    if start >= LOCAL_TEST_START and (not (ARTIFACTS / 'frozen_selection.json').is_file()):
        raise ValueError('Сначала зафиксируйте выбор по val')
    folder = RUN / start.strftime('%Y%m%d')
    folder.mkdir(parents=True, exist_ok=True)
    outputs = [folder / f'h{h}.parquet' for h in HORIZONS]
    if not all((path.is_file() for path in outputs)):
        lag_table = fit_lags(start)
        minute = times.hour * 60 + times.minute
        for h, path in zip(HORIZONS, outputs):
            if path.is_file():
                continue
            spec = selected.loc[h]
            if bool(spec.trimmed):
                raise ValueError('Для этого рецепта RF ожидается полная обучающая история')
            train_origins = np.flatnonzero(np.isin(minute, TRAIN_MINUTES) & (times + h * STEP < start))
            query_origins = np.flatnonzero((minute == 630) & (times >= start) & (times + h * STEP < stop))
            train, train_x = rf_data(h, train_origins, lag_table, spec)
            query, query_x = rf_data(h, query_origins, lag_table, spec)
            if bool(spec.route_median):
                median = train_x.groupby(train.route_id).median()
                train_x = train_x.fillna(median.reindex(train.route_id).set_axis(train_x.index))
                query_x = query_x.fillna(median.reindex(query.route_id).set_axis(query_x.index))
            imputer = SimpleImputer(strategy='median', keep_empty_features=True)
            train_matrix = imputer.fit_transform(train_x).astype(np.float32, copy=False)
            query_matrix = imputer.transform(query_x).astype(np.float32, copy=False)
            del train_x, query_x
            model = fit_forest(train, train_matrix, best_params[h])
            train_pred = predict_matrix(model, train, train_matrix)
            query_pred = predict_matrix(model, query, query_matrix)
            fit_report('rf', start, h, 'all', split, train.y, train_pred, query.y, query_pred)
            save_parquet(path, query[['cutoff', 'route_id', 'y']].assign(horizon=h, rf=query_pred))
            del model, train_matrix, query_matrix, train, query, train_pred, query_pred
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()
    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)
''', 'rf_runtime', 'exec'), rf_runtime)


## Обычный XGBoost на очищенной истории

Добавляем отдельный прогноз XGBoost MAE без кластеризации. Используем прежние признаки, параметры и число деревьев по горизонтам. Причинный MAD-4 фильтр очищает цель и шесть статусов по истории того же получасового слота. Лаги переоцениваются до каждой границы обучения. Обучаем на всех получасовых стартах, метрики считаем по исходным фактам из 10:30.

Для общего бленда проверяем варианты с этой моделью и без неё, а также пару Session6 + XGBoost clean. Прогноз входит в смесь без отдельного калибратора. Для сравнения при фиксированном объёме все варианты используют объём Session6. Новых входных датасетов не требуется.


In [ ]:
clean_xgb_runtime = dict(globals())
exec(compile(r'''

import xgboost as xgb
import cupy as cp

RUN = ARTIFACTS / 'xgb_clean'
RUN.mkdir(exist_ok=True)
CLEAN_XGB_CONTEXT = None


TARGET_SCALE = 100000.0
OBJECTIVE = 'reg:absoluteerror'
CLEANING = {'window_days': 28, 'min_days': 14, 'z_threshold': 4.0, 'log_scale_floor': 0.25, 'mad_normalization': 1.4826}
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
BEST_PARAMS = {'1': {'n_estimators': 2530, 'max_depth': 6, 'learning_rate': 0.036760715868196514, 'min_child_weight': 1.0809355847970479, 'subsample': 0.9003563537429966, 'colsample_bytree': 0.6639976846136139, 'reg_alpha': 9.305271303855188e-05, 'reg_lambda': 7.596418498005285}, '2': {'n_estimators': 1419, 'max_depth': 4, 'learning_rate': 0.10190438495585248, 'min_child_weight': 1.8389812995413264, 'subsample': 0.6817515181500764, 'colsample_bytree': 0.6619351860505892, 'reg_alpha': 0.0022965441587741537, 'reg_lambda': 14.08596091970978}, '3': {'n_estimators': 1930, 'max_depth': 3, 'learning_rate': 0.07005359250881395, 'min_child_weight': 2.2064897603579636, 'subsample': 0.8779096005107611, 'colsample_bytree': 0.7846752865583575, 'reg_alpha': 0.018954349619526665, 'reg_lambda': 1.043977626067855}, '4': {'n_estimators': 1827, 'max_depth': 4, 'learning_rate': 0.02925325002470326, 'min_child_weight': 1.5841239345284504, 'subsample': 0.8126045473410459, 'colsample_bytree': 0.7066552862291448, 'reg_alpha': 2.3192129195348187e-05, 'reg_lambda': 0.22202924731915144}, '5': {'n_estimators': 1387, 'max_depth': 3, 'learning_rate': 0.03164978915749773, 'min_child_weight': 1.9902815543275585, 'subsample': 0.6844098448217815, 'colsample_bytree': 0.7432566543439275, 'reg_alpha': 0.010252186188681373, 'reg_lambda': 0.5427730704771981}, '6': {'n_estimators': 1566, 'max_depth': 3, 'learning_rate': 0.023401679328514332, 'min_child_weight': 3.1806793320017555, 'subsample': 0.87035767368126, 'colsample_bytree': 0.7980364720685608, 'reg_alpha': 0.02123901077609503, 'reg_lambda': 0.3328783942599951}, '7': {'n_estimators': 1136, 'max_depth': 5, 'learning_rate': 0.0330759240922086, 'min_child_weight': 3.8692684642092345, 'subsample': 0.6794525496919193, 'colsample_bytree': 0.6799389555259473, 'reg_alpha': 2.7314479492664567, 'reg_lambda': 0.3796668381307971}, '8': {'n_estimators': 534, 'max_depth': 4, 'learning_rate': 0.09417045025581995, 'min_child_weight': 13.418857993305474, 'subsample': 0.689895840619612, 'colsample_bytree': 0.6918995986906583, 'reg_alpha': 0.3746751674810787, 'reg_lambda': 1.8540896283895405}}

def clean_raw_array(array, timestamps, settings):
    array = np.asarray(array, dtype=np.float32)
    if not np.isfinite(array).all() or (array < 0).any():
        raise ValueError('Для log1p нужны конечные неотрицательные исходные значения')
    cleaned = array.copy()
    flagged = np.zeros(array.shape, dtype=bool)
    slots = timestamps.hour * 2 + timestamps.minute // 30
    for slot in range(48):
        positions = np.flatnonzero(slots == slot)
        logged = np.log1p(array[positions].astype(np.float64))
        window_days = settings['window_days']
        min_days = settings['min_days']
        if not 1 <= min_days <= window_days:
            raise ValueError('Нужно 1 <= min_days <= window_days')
        if len(positions) <= min_days:
            continue
        padded = np.pad(logged, ((window_days, 0), (0, 0)), constant_values=np.nan)
        windows = np.lib.stride_tricks.sliding_window_view(padded, window_days, axis=0)[:-1]
        eligible = np.arange(len(positions)) >= min_days
        past = windows[eligible]
        median = np.nanmedian(past, axis=-1)
        mad = np.nanmedian(np.abs(past - median[..., None]), axis=-1)
        scale = np.maximum(settings['mad_normalization'] * mad, settings['log_scale_floor'])
        mask = np.abs(logged[eligible] - median) > settings['z_threshold'] * scale
        destination = positions[eligible]
        cleaned[destination] = np.where(mask, np.expm1(median), array[destination]).astype(np.float32)
        flagged[destination] = mask
    return (cleaned, flagged)

FEATURE_CODE = "import pyarrow as pa\nimport pyarrow.parquet as pq\ntarget = values[TARGET]\n\ndef past(array, positions):\n    positions = np.asarray(positions, dtype=np.int64)\n    valid = (positions >= 0) & (positions < len(array))\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\n    result[~valid] = np.nan\n    return result\n\ndef average_available(matrices):\n    total = np.zeros_like(matrices[0], dtype=np.float32)\n    count = np.zeros_like(total)\n    for matrix in matrices:\n        valid = np.isfinite(matrix)\n        total += np.where(valid, matrix, 0)\n        count += valid\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\n\ndef rolling(array, steps):\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\nlevel24 = rolling(target, 48)\npositions = np.arange(len(times))\nextra_signals = {}\nfor name in STATUSES[:3]:\n    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)\nextra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))\nextra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))\nfor name in STATUSES[3:]:\n    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)\nextra_signals['hub_buffer'] = values['status_5'] - values['status_4']\nextra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']\nextra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)\nhub_signal = extra_signals['status_6_diff_hourly']\n\ndef common_features(origins):\n    common = {}\n    for k in (0, 1, 2, 4):\n        common[f'target_back_{k}'] = past(target, origins - k)\n    for name, array in values.items():\n        for steps in (4, 12):\n            common[f'{name}_mean_{steps}'] = means[name, steps][origins]\n        if name in STATUSES:\n            common[f'{name}_latest'] = array[origins]\n            if name in STATUSES[:3]:\n                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)\n    for name, array in extra_signals.items():\n        common[name] = array[origins]\n    for lag in (48, 96):\n        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)\n    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)\n    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)\n    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]\n    recent = means[TARGET, 4][origins]\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\n    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\n    return common\nCORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']\nFEATURE_GROUPS = {'calendar': ['hour', 'minute', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': ['status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'], 'hub_combinations': ['status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'], 'historical_lags': ['target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48'], 'target_change': ['target_acceleration'], 'correlation_lag': CORRELATION_COLUMNS, 'pilot_start_time': ['start_sin', 'start_cos'], 'pilot_peaks': ['peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']}\n\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\n    grid = pd.DatetimeIndex(timestamps)\n    route_ids = np.asarray(route_ids)\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\n        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\n        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')\n    if len(np.unique(route_ids)) != len(route_ids):\n        raise ValueError('Идентификаторы маршрутов повторяются')\n    steps = float(max_lookback_hours) * 2\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\n        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\n        raise ValueError('min_pairs должен быть целым числом >=3')\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\n        raise ValueError('Некорректные параметры выбора корреляции')\n    max_lag = int(steps)\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))\n    anchors = np.arange(max_lag + 2, stop)\n    best_corr = np.full(len(route_ids), np.nan)\n    best_lag = np.full(len(route_ids), np.nan)\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\n    for start in range(0, len(route_ids), route_block_size):\n        finish = min(start + route_block_size, len(route_ids))\n        x = target_values[anchors, start:finish].astype(np.float64)\n        for lag in range(2, max_lag + 1):\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\n            valid = np.isfinite(x) & np.isfinite(z)\n            count = valid.sum(axis=0)\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\n            divisor = np.maximum(count, 1)\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\n            denominator = np.sqrt(vx * vz)\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\n            corr = np.clip(corr, -1, 1)\n            previous = best_corr[start:finish]\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\n            best_corr[start:finish][better] = corr[better]\n            best_lag[start:finish][better] = lag\n            best_pairs[start:finish][better] = count[better]\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\n    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})\n\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\n    if h not in range(1, 9) or chunk_rows < 1:\n        raise ValueError('Некорректный горизонт или размер блока')\n    by_route = lag_table.set_index('route_id').reindex(routes)\n    route_index = pd.Index(routes)\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\n    for name in CORRELATION_COLUMNS:\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\n    for start in range(0, len(frame), chunk_rows):\n        finish = min(start + chunk_rows, len(frame))\n        block = frame.iloc[start:finish]\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\n        route = route_index.get_indexer(block.route_id)\n        if (origin < 0).any() or (route < 0).any():\n            raise ValueError('Строка признаков не сопоставлена с историей')\n        selected = lag[route]\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\n        available &= np.isfinite(signal)\n        for name, array in quality.items():\n            frame.iloc[start:finish, column_positions[name]] = array[route]\n        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)\n        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)\n    return frame\n\ndef horizon_dataset(h, origins):\n    slot = origins + h\n    dates = times[origins] + h * STEP\n    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})\n    for name, matrix in common_features(origins).items():\n        frame[name] = matrix.ravel()\n    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}\n    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)\n    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)\n    for name, array in calendar.items():\n        frame[name] = np.repeat(array, len(routes))\n    frame = add_correlation_features(frame, h, validation_lag_table)\n    for days in (1, 2, 7):\n        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()\n    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\n    profile = np.zeros((len(origins), len(routes)))\n    counts = np.zeros_like(profile)\n    for day in range(1, 29):\n        level = past(level24, origins - day * 48)\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\n        profile += np.nan_to_num(adjusted)\n        counts += np.isfinite(adjusted)\n    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\n    if h == 1:\n        lower = np.zeros_like(target[origins])\n        upper = past(target, origins - 2)\n        for position in (origins - 1, origins):\n            total = past(target, position)\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\n        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()\n        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\n    else:\n        frame['overlap'] = np.nan\n    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])\n    return frame.astype({name: 'float32' for name in numeric})\nbase_horizon_dataset = horizon_dataset\nnetwork_values = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None] for name, array in values.items() if name in [TARGET, *STATUSES[:3]]}\n\ndef ratio_features(current, usual):\n    denominator = np.maximum(usual, np.float32(1.0))\n    ratio = np.divide(current, denominator, out=np.full_like(current, np.nan), where=np.isfinite(usual))\n    return (ratio, np.where(np.isfinite(usual), usual < 1.0, np.nan).astype(np.float32))\n\ndef horizon_dataset(h, origins):\n    frame = base_horizon_dataset(h, origins)\n    added = {}\n    slot = origins + h\n    profile = {}\n    for offset in (-2, -1, 0, 1, 2):\n        profile[offset] = average_available([past(target, slot + offset - day * 48) for day in range(1, 8)])\n        if offset != 0:\n            added[f'slot_daily7_neighbor_{offset:+d}'] = profile[offset].ravel()\n    added['slot_daily7_change_2h'] = (profile[2] - profile[-2]).ravel()\n    stack = np.stack([profile[offset] for offset in (-2, -1, 0, 1, 2)])\n    finite = np.isfinite(stack)\n    masked = np.where(finite, stack, -np.inf)\n    peak = masked.max(axis=0)\n    available = finite.any(axis=0)\n    added['slot_daily7_peak_value'] = np.where(available, peak, np.nan).ravel()\n    added['slot_daily7_peak_offset_minutes'] = np.where(available, (masked.argmax(axis=0) - 2) * 30, np.nan).ravel()\n    for name in STATUSES[:3]:\n        for suffix, array in [('latest', values[name]), ('mean_4', means[name, 4])]:\n            current = array[origins]\n            usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\n            ratio, small = ratio_features(current, usual)\n            added[f'{name}_{suffix}_vs_usual7'] = (current - usual).ravel()\n            added[f'{name}_{suffix}_ratio_usual7'] = ratio.ravel()\n            added[f'{name}_{suffix}_usual7_small'] = small.ravel()\n    for day in (14, 21, 28):\n        added[f'slot_days_back_{day}'] = past(target, slot - day * 48).ravel()\n    added['slot_week1_minus_week2'] = (past(target, slot - 7 * 48) - past(target, slot - 14 * 48)).ravel()\n    recent = average_available([past(target, slot - day * 48) for day in (1, 2, 3)])\n    previous = average_available([past(target, slot - day * 48) for day in (4, 5, 6, 7)])\n    added['slot_recent3_minus_previous4'] = (recent - previous).ravel()\n    for name, array in network_values.items():\n        current = array[origins]\n        usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\n        ratio, small = ratio_features(current, usual)\n        features = {'latest': current, 'change_1h': current - past(array, origins - 2), 'vs_usual7': current - usual, 'ratio_usual7': ratio, 'usual7_small': small}\n        for suffix, feature in features.items():\n            added[f'network_{name}_{suffix}'] = np.broadcast_to(feature, (len(origins), len(routes))).ravel()\n        local = values[name][origins]\n        share = np.divide(local, current, out=np.full_like(local, np.nan), where=current > 0)\n        past_total = past(array, origins - 2)\n        old_share = np.divide(past(values[name], origins - 2), past_total, out=np.full_like(local, np.nan), where=past_total > 0)\n        added[f'network_{name}_route_share'] = share.ravel()\n        added[f'network_{name}_route_share_change_1h'] = (share - old_share).ravel()\n    return pd.concat([frame, pd.DataFrame(added, index=frame.index, dtype=np.float32)], axis=1)"

class CleanXgbBatches(xgb.DataIter):

    def __init__(self, matrix, labels, columns):
        self.matrix = matrix
        self.labels = labels
        self.columns = columns
        self.types = ['c' if column == 'route_id' else 'q' for column in columns]
        self.position = 0
        super().__init__(release_data=True)

    def reset(self):
        self.position = 0

    def next(self, input_data):
        if self.position >= len(self.labels):
            return False
        start = self.position
        stop = min(start + 65536, len(self.labels))
        self.position = stop
        input_data(data=cp.asarray(self.matrix[start:stop]), label=cp.asarray(self.labels[start:stop]), feature_names=self.columns, feature_types=self.types)
        return True

def clean_xgb_matrix(h, origins, features):
    route_count = len(ROUTES)
    route_codes = np.arange(route_count, dtype=np.float32)
    matrix = None
    for start in range(0, len(origins), 24):
        block = origins[start:start + 24]
        frame = features(h, block)
        columns = frame.columns.drop(['cutoff', 'timestamp', 'overlap']).tolist()
        frame['route_id'] = np.tile(route_codes, len(block))
        if matrix is None:
            names = columns
            matrix = np.empty((len(origins) * route_count, len(names)), dtype=np.float32)
        if columns != names:
            raise ValueError('Расходятся колонки обычного XGBoost')
        matrix[start * route_count:(start + len(block)) * route_count] = frame[columns].to_numpy(dtype=np.float32)
    if matrix is None:
        raise ValueError('Пустая матрица обычного XGBoost')
    return (matrix, names)

def predict_clean_xgb(model, matrix):
    result = np.empty(len(matrix), dtype=np.float64)
    for start in range(0, len(matrix), 65536):
        stop = min(start + 65536, len(matrix))
        result[start:stop] = cp.asnumpy(model.inplace_predict(cp.asarray(matrix[start:stop]))) * TARGET_SCALE
    return np.maximum(result, 0)

def clean_xgb_features():
    global CLEAN_XGB_CONTEXT
    if CLEAN_XGB_CONTEXT is None:
        values = {}
        rows = []
        for name in [TARGET, *STATUSES]:
            values[name], flagged = clean_raw_array(VALUES[name], TIMES, CLEANING)
            rows.append(dict(column=name, replaced=int(flagged.sum()), rows=flagged.size))
        pd.DataFrame(rows).to_csv(RUN / 'cleaning_counts.csv', index=False)
        CLEAN_XGB_CONTEXT = dict(np=np, pd=pd, TARGET=TARGET, STATUSES=STATUSES, STEP=STEP, values=values, times=TIMES, routes=ROUTES)
        exec(FEATURE_CODE, CLEAN_XGB_CONTEXT)
    return CLEAN_XGB_CONTEXT

def fit_clean_xgb(start, stop, split):
    if start >= LOCAL_TEST_START and (not (ARTIFACTS / 'frozen_selection.json').is_file()):
        raise ValueError('Сначала зафиксируйте выбор по val')
    folder = RUN / start.strftime('%Y%m%d')
    folder.mkdir(parents=True, exist_ok=True)
    outputs = [folder / f'h{h}.parquet' for h in HORIZONS]
    if not all((path.is_file() for path in outputs)):
        features = clean_xgb_features()
        target = features['target']
        lag_table = features['calculate_dynamic_shoulder'](target, features['hub_signal'], TIMES, ROUTES, fit_before=start, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
        features['validation_lag_table'] = lag_table
        lag_table.to_csv(folder / 'status6_lags.csv', index=False)
        for h, path in zip(HORIZONS, outputs):
            if path.is_file():
                continue
            origins = np.arange(len(TIMES) - h)
            train = origins[TIMES[origins + h] < start]
            query = origins[(TIMES[origins] >= start) & (TIMES[origins + h] < stop) & (TIMES[origins].hour == 10) & (TIMES[origins].minute == 30)]
            print('XGBoost clean', split, start.date(), f'h{h}', flush=True)
            matrix, columns = clean_xgb_matrix(h, train, features['horizon_dataset'])
            labels = (target[train + h].ravel() / TARGET_SCALE).astype(np.float32)
            batches = CleanXgbBatches(matrix, labels, columns)
            quantile = xgb.QuantileDMatrix(batches, max_bin=256, max_quantile_batches=8, enable_categorical=True)
            params = BEST_PARAMS[str(h)].copy()
            trees = params.pop('n_estimators')
            params.update(objective=OBJECTIVE, eval_metric='mae', tree_method='hist', device='cuda:0', max_bin=256, max_cat_to_onehot=1, seed=SEED, nthread=THREADS)
            model = xgb.train(params, quantile, num_boost_round=trees)
            device = json.loads(model.save_config())['learner']['generic_param']['device']
            if not device.startswith('cuda') or model.feature_types != batches.types:
                raise RuntimeError('XGBoost clean потерял CUDA или категориальный route_id')
            del quantile, batches
            cp.get_default_memory_pool().free_all_blocks()
            train_prediction = predict_clean_xgb(model, matrix)
            del matrix
            query_matrix, query_columns = clean_xgb_matrix(h, query, features['horizon_dataset'])
            if columns != query_columns:
                raise ValueError('Расходятся колонки train и проверки обычного XGBoost')
            prediction = predict_clean_xgb(model, query_matrix)
            actual = VALUES[TARGET][query + h].ravel()
            fit_report('xgb_clean', start, h, 'all', split, VALUES[TARGET][train + h].ravel(), train_prediction, actual, prediction)
            frame = pd.DataFrame(dict(cutoff=TIMES[query].repeat(len(ROUTES)), route_id=np.tile(ROUTES, len(query)), horizon=h, y=actual, xgb_clean=prediction))
            save_parquet(path, frame)
            del model, query_matrix, train_prediction
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()
    return pd.concat([pd.read_parquet(path) for path in outputs], ignore_index=True)
''', 'clean_xgb_runtime', 'exec'), clean_xgb_runtime)


## Сборка OOF

Для XGBoost rich берём четыре недели готовых raw-прогнозов. Читаем файлы прямо из Kaggle Dataset. Сверяем хеши данных, границы, ключи, факты и время каждой цели. Метапризнаки из истории используют только наблюдения, доступные на момент прогноза. `profile28` строится из предыдущих 28 дней с поправкой на текущий масштаб маршрута. Базовые метрики на OOF и test будут собраны в итоговой таблице.


In [ ]:
session_oof = session_runtime['session6_oof']()
for name in ['means', 'level24', 'extra_signals', 'hub_signal', 'long_week', 'deviations', 'validation_lag_table']:
    session_runtime.pop(name, None)
oof_parts = []
for index, (start, stop) in enumerate(OOF_WINDOWS):
    stage = f'oof_week_{index}' if index < 3 else 'calibration_select'
    xgb = read_xgb(stage)
    cat = cat_runtime['fit_rich_cat'](start, stop, 'oof')
    rf = rf_runtime['fit_rf'](start, stop, 'oof')
    xgb_clean = clean_xgb_runtime['fit_clean_xgb'](start, stop, 'oof')
    oof_parts.append(assemble(start, stop, session_oof, rf, cat, xgb, xgb_clean))
    gc.collect()
oof = pd.concat(oof_parts, ignore_index=True)
save_parquet(ARTIFACTS / 'base_oof.parquet', oof)
display(pd.DataFrame([dict(model=name, **metric(oof.y, oof[name])) for name in BASES]))


## Кандидаты метамодели

Проверяем Session6, равные веса, парные бленды, общий бленд с XGBoost clean и без него, а также усаженные веса по горизонтам. Ridge, CatBoost MAE и XGB MAE учатся предсказывать остаток общего бленда. Gating Network выдаёт для каждой строки веса моделей, которые складываются в единицу. Начальная точка сети берётся из общего бленда, а штраф ограничивает отход от него. Все поправки формы возвращаются к сумме Session6.

Входы включают прогнозы в масштабе маршрута, их отклонения от Session6, разброс, горизонт, день недели, текущий уровень, изменения за 1 и 2 часа, средний уровень маршрута за 24 часа и текущий объём сети. Сезонный `profile28` участвует как прогноз и признак. Отдельная поправка объёма сравнивает фиксированную сумму, усаженные общий и свежий коэффициенты и Ridge по агрегатам. Множители объёма ограничены ±5%.

На 4–10 октября метамодели проверяются после fit на первых двух OOF-неделях. На 11–17 октября используются первые три. Победитель определяется по среднему WAPE + |Relative Bias| двух недель. После фиксации схемы метамодели обучаются на четырёх OOF-неделях. На test публикуются результаты всех заранее заданных вариантов, включая победителя val.


In [ ]:
import joblib
import torch
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


PAIRS = BASES[1:]
VOLUME_INPUTS = [name for name in PAIRS if name != 'xgb_clean']
SHAPES = ['session6', 'equal', 'joint', 'joint_without_xgb_clean', 'by_horizon',
          *[f'pair_{name}' for name in PAIRS],
          'ridge_100', 'ridge_1000', 'cat_d2', 'cat_d4', 'xgb_d2', 'xgb_d3', 'gating_8', 'gating_16']
VOLUMES = ['fixed', 'ratio', 'ewma7', 'ridge']
META_DEVICE = 'cuda'


def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out


def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])


def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights


def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level


class GatingNetwork(torch.nn.Module):
    def __init__(self, width, experts, hidden, prior):
        super().__init__()
        self.layers = torch.nn.Sequential(torch.nn.Linear(width, hidden), torch.nn.Tanh(),
                                          torch.nn.Linear(hidden, experts))
        self.register_buffer('prior', torch.as_tensor(prior, dtype=torch.float32))
        torch.nn.init.zeros_(self.layers[-1].weight)
        torch.nn.init.zeros_(self.layers[-1].bias)

    def forward(self, context):
        return torch.softmax(self.prior.log() + self.layers(context), dim=1)


def fit_gating(x, experts, y, level, prior, hidden):
    torch.manual_seed(SEED)
    scaler = StandardScaler().fit(x)
    prior = (prior + .02) / (prior.sum() + .02 * len(prior))
    model = GatingNetwork(x.shape[1], experts.shape[1], hidden, prior).to(META_DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=.01)
    normalized = experts / level[:, None]
    target = np.asarray(y, dtype=np.float64) / level
    weights = level / level.mean()
    tensors = [torch.as_tensor(value, dtype=torch.float32, device=META_DEVICE)
               for value in (scaler.transform(x), normalized, target, weights)]
    tx, te, ty, tw = tensors
    for _ in range(15):
        permutation = torch.randperm(len(x), device=META_DEVICE)
        for rows in permutation.split(4096):
            gate = model(tx[rows])
            pred = (gate * te[rows]).sum(dim=1)
            loss = ((pred - ty[rows]).abs() * tw[rows]).mean() + .03 * (gate - model.prior).square().sum(dim=1).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return scaler, model.cpu()


def fit_shape(name, frame, matrix, joint):
    if name == 'session6':
        return None
    if name == 'equal':
        return np.full(len(BASES), 1 / len(BASES))
    if name == 'joint':
        return joint
    if name == 'joint_without_xgb_clean':
        mask = np.array([base != 'xgb_clean' for base in BASES])
        weights = np.zeros(len(BASES))
        weights[mask] = fit_weights(matrix[:, mask], frame.y)
        return weights
    if name.startswith('pair_'):
        index = BASES.index(name.removeprefix('pair_'))
        weights = np.zeros(len(BASES))
        weights[[0, index]] = fit_weights(matrix[:, [0, index]], frame.y)
        return weights
    if name == 'by_horizon':
        weights = []
        for h in HORIZONS:
            mask = frame.horizon.eq(h).to_numpy()
            weights.append(.5 * joint + .5 * fit_weights(matrix[mask], frame.y.to_numpy()[mask]))
        return np.stack(weights)
    baseline = matrix @ joint
    features, level = meta_features(frame, matrix, baseline)
    residual = (frame.y.to_numpy(dtype=np.float64) - baseline) / level
    sample_weight = level / level.mean()
    if name.startswith('gating_'):
        return fit_gating(features, matrix, frame.y, level, joint, int(name.split('_')[-1]))
    if name.startswith('ridge_'):
        prep = StandardScaler().fit(features)
        model = Ridge(alpha=float(name.split('_')[-1]))
        model.fit(prep.transform(features), residual, sample_weight=sample_weight)
        return prep, model
    depth = int(name[-1])
    if name.startswith('cat_'):
        model = CatBoostRegressor(depth=depth, iterations=220, loss_function='MAE', learning_rate=.035,
                                  l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                                  thread_count=THREADS)
    else:
        model = XGBRegressor(max_depth=depth, n_estimators=220, objective='reg:absoluteerror',
                             tree_method='hist', learning_rate=.035, reg_lambda=20,
                             min_child_weight=30, random_state=SEED, n_jobs=THREADS)
    model.fit(features, residual, sample_weight=sample_weight)
    return None, model


def predict_shape(name, fitted, frame, matrix, joint):
    if name == 'session6':
        return matrix[:, 0]
    if isinstance(fitted, np.ndarray):
        if fitted.ndim == 1:
            return matrix @ fitted
        return np.sum(matrix * fitted[frame.horizon.to_numpy() - 1], axis=1)
    baseline = matrix @ joint
    prep, model = fitted
    features, level = meta_features(frame, matrix, baseline)
    if name.startswith('gating_'):
        weights = gate_weights(prep, model, features)
        return with_volume(frame, np.sum(matrix * weights, axis=1))
    encoded = features if prep is None else prep.transform(features)
    correction = np.clip(model.predict(encoded), -.5, .5) * level
    return with_volume(frame, baseline + correction)


def gate_weights(prep, model, features):
    model.eval()
    with torch.no_grad():
        return model(torch.as_tensor(prep.transform(features), dtype=torch.float32)).numpy()


def volume_table(frame, with_y=False):
    aggregations = {name: (name, 'sum') for name in BASES}
    aggregations['network_current'] = ('network_current', 'first')
    if with_y:
        aggregations['y'] = ('y', 'sum')
    return frame.groupby(GROUP_KEYS, as_index=False, sort=True).agg(**aggregations)


def volume_features(table):
    anchor = np.maximum(table.session6.to_numpy(), 1)
    features = np.column_stack([np.log1p(anchor), np.log1p(table.network_current),
                                table.horizon, table.cutoff.dt.day_of_week,
                                *[np.log((table[name].to_numpy() + 1) / (anchor + 1)) for name in VOLUME_INPUTS]])
    return features


def fit_volumes(frame):
    table = volume_table(frame, with_y=True)
    if table.session6.sum() <= 0:
        raise ValueError('Нулевой объём Session6')
    ages = (table.cutoff.max() - table.cutoff).dt.total_seconds().to_numpy() / 86400
    w = np.exp2(-ages / 7)
    ratio = table.y.sum() / table.session6.sum()
    recent_ratio = np.dot(table.y, w) / np.dot(table.session6, w)
    x = volume_features(table)
    prep = StandardScaler().fit(x)
    model = Ridge(alpha=100).fit(prep.transform(x), np.log((table.y + 1) / (table.session6 + 1)))
    return dict(fixed=1., ratio=float(1 + .5 * (ratio - 1)),
                ewma7=float(1 + .5 * (recent_ratio - 1)), ridge=(prep, model))


def volume_multipliers(frame, fitted):
    table = volume_table(frame)
    prep, model = fitted['ridge']
    fitted = dict(fitted, ridge=1 + .5 * np.expm1(model.predict(prep.transform(volume_features(table)))))
    return {name: frame[GROUP_KEYS].merge(table[GROUP_KEYS].assign(factor=np.clip(factor, .95, 1.05)),
                                           on=GROUP_KEYS, how='left', validate='many_to_one').factor.to_numpy()
            for name, factor in fitted.items()}


def fit_meta(train, query, phase, split):
    train_matrix, query_matrix = shape_matrix(train), shape_matrix(query)
    joint = fit_weights(train_matrix, train.y)
    volume_models = fit_volumes(train)
    train_factors = volume_multipliers(train, volume_models)
    query_factors = volume_multipliers(query, volume_models)
    rows, fitted, predictions = [], {}, {}
    for name in SHAPES:
        model = fit_shape(name, train, train_matrix, joint)
        fitted[name] = model
        train_prediction = predict_shape(name, model, train, train_matrix, joint)
        query_prediction = predict_shape(name, model, query, query_matrix, joint)
        local = []
        for volume in VOLUMES:
            label = f'{name}__{volume}'
            train_values = train_prediction * train_factors[volume]
            query_values = query_prediction * query_factors[volume]
            predictions[label] = query_values
            local.extend([dict(phase=phase, variant=label, shape=name, volume=volume, period=period, **metric(y, p))
                          for period, y, p in [('train_oof', train.y, train_values), (split, query.y, query_values)]])
        display(pd.DataFrame(local))
        rows.extend(local)
    return pd.DataFrame(rows), dict(joint=joint, shapes=fitted, volumes=volume_models), predictions


def fit_selection(oof):
    blocks = []
    for fold, (start, stop) in enumerate(OOF_WINDOWS[2:], 3):
        train = oof.loc[oof.cutoff.lt(start)]
        validation = oof.loc[oof.cutoff.ge(start) & oof.cutoff.lt(stop)]
        rows, _, _ = fit_meta(train, validation, f'val_week_{fold}', 'val')
        blocks.append(rows)
    metrics = pd.concat(blocks, ignore_index=True)
    metrics.to_csv(ARTIFACTS / 'selection_metrics.csv', index=False)
    ranking = metrics.loc[metrics.period.eq('val')].groupby(['variant', 'shape', 'volume'], as_index=False).agg(
        mean_val_score=('score', 'mean'), worst_val_score=('score', 'max'), weeks=('score', 'count'))
    ranking['order'] = ranking['shape'].map({name: i for i, name in enumerate(SHAPES)})
    ranking = ranking.sort_values(['mean_val_score', 'worst_val_score', 'order', 'variant'])
    if not ranking.weeks.eq(2).all():
        raise ValueError('В подборе должны присутствовать обе недели val')
    ranking.to_csv(ARTIFACTS / 'selection_ranking.csv', index=False)
    display(ranking.drop(columns='order').head(20))
    selected = ranking.iloc[0]
    frozen = dict(variant=selected.variant, shape=selected['shape'], volume=selected.volume,
                  mean_val_score=float(selected.mean_val_score), test_used_for_selection=False,
                  meta_fit_weeks=4, local_test_before_full_refit=True)
    write_json(ARTIFACTS / 'frozen_selection.json', frozen)
    return frozen


def evaluate(oof, test, frozen):
    rows, models, predictions = fit_meta(oof, test, 'frozen_local_test', 'test')
    base_rows = [dict(phase='base', variant=column, shape=column, volume='raw', period=period,
                      **metric(frame.y, frame[column]))
                 for period, frame in [('train_oof', oof), ('test', test)] for column in BASES]
    table = pd.concat([pd.DataFrame(base_rows), rows], ignore_index=True)
    table['selected_on_val'] = table.variant.eq(frozen['variant'])
    table.to_csv(ARTIFACTS / 'train_test_metrics.csv', index=False)
    predictions = pd.concat([test.reset_index(drop=True), pd.DataFrame(predictions)], axis=1)
    save_parquet(ARTIFACTS / 'test_predictions.parquet', predictions)
    columns = [*BASES, *[f'{name}__fixed' for name in SHAPES]]
    if frozen['variant'] not in columns:
        columns.append(frozen['variant'])
    slices = []
    for kind, key in [('day', predictions.cutoff), ('horizon', predictions.horizon),
                      ('week', ((predictions.cutoff.dt.normalize() - LOCAL_TEST_START).dt.days // 7) + 1)]:
        for value, group in predictions.groupby(key):
            slices.extend(dict(slice=kind, value=str(value), variant=name, **metric(group.y, group[name])) for name in columns)
    pd.DataFrame(slices).to_csv(ARTIFACTS / 'test_slices.csv', index=False)
    weights = []
    matrix = shape_matrix(test)
    features, _ = meta_features(test, matrix, matrix @ models['joint'])
    for name, model in models['shapes'].items():
        if isinstance(model, np.ndarray):
            if model.ndim == 1:
                weights.append(dict(variant=name, horizon='all', **dict(zip(BASES, model))))
            else:
                weights.extend(dict(variant=name, horizon=h, **dict(zip(BASES, row)))
                               for h, row in zip(HORIZONS, model))
        elif name.startswith('gating_'):
            prep, network = model
            frame = pd.DataFrame(gate_weights(prep, network, features), columns=BASES)
            means = frame.groupby(test.horizon.to_numpy()).mean()
            weights.extend(dict(variant=name, horizon=h, **row.to_dict()) for h, row in means.iterrows())
            models['shapes'][name] = dict(prep=prep, state=network.state_dict(),
                                         width=features.shape[1], hidden=int(name.split('_')[-1]))
    pd.DataFrame(weights).to_csv(ARTIFACTS / 'blend_weights.csv', index=False)
    artifact = ARTIFACTS / 'evaluation_meta.joblib'
    temporary = artifact.with_suffix('.partial')
    joblib.dump(dict(frozen=frozen, bases=BASES, models=models), temporary)
    temporary.replace(artifact)
    comparison = ['joint__fixed', 'joint_without_xgb_clean__fixed', 'pair_xgb_clean__fixed']
    display(table.loc[table.selected_on_val | table.variant.isin([*BASES, *comparison])])
    note = ('Независимая от полного переобучения оценка test получена до переобучения. '
            'Локальный период 18–31 октября уже изучался. Выбор сделан на двух более ранних неделях. '
            'Новые метамодели обучены только на OOF до test. '
            'Session6 на OOF восстановлен по рецепту, а на test взят из архива решения 0.326. '
            'Полное переобучение и конкурсный сабмит не выполняются.')
    (ARTIFACTS / 'evaluation_note.txt').write_text(note)
    print(note)


In [ ]:
frozen = fit_selection(oof)
display(pd.DataFrame([frozen]))


## Локальный test и отчёт

RF, CatBoost rich и XGBoost на очищенной истории переобучаются на целях строго до 18 октября. XGBoost rich берётся из сохранённого запуска с той же границей. Для каждой новой метамодели печатаются train OOF и локальный test. У train OOF базовые прогнозы вне обучающих окон, а метамодель оценивается на своём обучении. В отчёт также входят срезы по дням, неделям и горизонтам, выбор по val, построчные прогнозы и параметры метамоделей.

Независимая от полного переобучения оценка test получена до переобучения.


In [ ]:
if json.loads((ARTIFACTS / 'frozen_selection.json').read_text()) != frozen:
    raise ValueError('Сначала зафиксируйте выбор по val')
xgb_test = read_xgb('train_val')
session_test = read_session6_test()
cat_test = cat_runtime['fit_rich_cat'](LOCAL_TEST_START, LOCAL_TEST_END, 'test')
rf_test = rf_runtime['fit_rf'](LOCAL_TEST_START, LOCAL_TEST_END, 'test')
clean_test = clean_xgb_runtime['fit_clean_xgb'](LOCAL_TEST_START, LOCAL_TEST_END, 'test')
test = assemble(LOCAL_TEST_START, LOCAL_TEST_END, session_test, rf_test, cat_test, xgb_test, clean_test)
evaluate(oof, test, frozen)
archive_report()
